In [1]:
# ============================================================
# POV CREATOR - V0.9
# Google Colab + Gradio
# ============================================================
#
# DOCUMENTACIÓN TÉCNICA PARA TESIS
#
# Finalidad general
# -----------------
# Preparar contenido gráfico para una pantalla POV basada en ESP32.
# Concentrar en una sola interfaz las etapas de normalización de
# imágenes, creación de animaciones, adaptación de videos y generación
# de los archivos imgs.h utilizados por el firmware embebido.
#
# Relación software - hardware
# ----------------------------
# Convertir contenido visual convencional a una representación adecuada
# para visualización por persistencia de la visión. Trabajar con una
# superficie lógica de 200 x 200 píxeles y transformar posteriormente
# esa información cartesiana a muestras polares compatibles con el
# barrido angular del sistema físico.
#
# Resolución angular utilizada: 6 grados.
# Vueltas lógicas por imagen: 360 / 6 = 60 posiciones angulares.
# Información radial por posición: 100 valores.
# Formato de color final: RGB565 de 16 bits.
#
# La matriz 60 x 100 constituye la representación intermedia utilizada
# por el firmware para reconstruir la imagen durante la rotación.
#
# Flujo principal
# ---------------
# Imagen / video / secuencia de imágenes
#                  |
#                  v
#        Normalización 200 x 200
#                  |
#                  v
#       Conversión temporal / espacial
#                  |
#                  v
#             Generación imgs.h
#                  |
#                  v
#              Firmware ESP32
#                  |
#                  v
#             Visualización POV
#
# Restricción de memoria
# ----------------------
# Mantener un máximo de 69 frames almacenados para animación. Cada frame
# utiliza aproximadamente 12 000 bytes: 60 filas x 100 valores x 2 bytes.
# El límite permite trabajar dentro de la partición de aplicación cercana
# a 2 MB configurada para conservar actualización OTA.
#
# Criterio de integridad
# ----------------------
# Mantener sin alteraciones funcionales la geometría polar, el empaquetado
# RGB565 y las estructuras frame5[][100] y frames[][60][100]. Estos bloques
# representan la interfaz directa entre el software de preparación y el
# firmware del sistema POV.
#
# Módulos funcionales
# -------------------
#   - Crear Animación
#   - Adaptar video
#   - Adaptar imagen
#   - Generar imgs.h desde animación
#   - Generar imgs.h desde imagen
#
# Nota de mantenimiento
# ---------------------
# Conservar nombres internos como STOPMOTION_DIR, variables stop_* y rutas
# históricas para evitar cambios innecesarios en el comportamiento. El
# nombre visible de la herramienta pasa a ser CREAR ANIMACIÓN.
# ============================================================

import sys
import subprocess

# Instalar/actualizar Gradio desde Python para que este archivo
# también pueda ejecutarse con %run en Google Colab.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "gradio"],
    check=True
)

import gradio as gr
import os
import re
import cv2
import json
import math
import base64
import shutil
import tempfile
import mimetypes
import numpy as np

from PIL import Image, ImageOps
from google.colab import drive
from IPython.display import display, HTML


# ============================================================
# MONTAR GOOGLE DRIVE
# ============================================================
# Persistencia de archivos fuera de la sesión temporal de Colab.
# Mantener recursos gráficos, videos procesados, imágenes normalizadas,
# fondos de interfaz y archivos imgs.h dentro de una estructura común.
# Evitar dependencia del almacenamiento efímero del entorno de ejecución.

drive.mount("/content/drive")


# ============================================================
# CONFIGURACION POV
# ============================================================
# Parámetros comunes para toda la cadena de procesamiento.
# 200 x 200: formato cartesiano normalizado antes de conversión polar.
# 10 FPS: frecuencia de preparación de animaciones y videos.
# 69 frames: límite práctico de almacenamiento para animación con OTA.
# Fondo negro: ausencia de emisión luminosa en zonas transparentes o libres.

ANCHO = 200
ALTO = 200
FPS = 10
TOTAL_FRAMES_FINAL = 69
COLOR_FONDO = (0, 0, 0)


# ============================================================
# RUTAS
# ============================================================
# Separación entre almacenamiento permanente y procesamiento temporal.
# Drive: resultados reutilizables entre sesiones.
# TEMP_ROOT: archivos intermedios descartables durante cada conversión.
# La organización por carpetas facilita trazabilidad y repetición de pruebas.

BASE_DIR = "/content/drive/MyDrive/TESIS_POV"
PROGRAMA_DIR = os.path.join(BASE_DIR, "Programa")
ASSETS_DIR = os.path.join(PROGRAMA_DIR, "assets")
VIDEOS_DIR = os.path.join(BASE_DIR, "Videos")
STOPMOTION_DIR = os.path.join(BASE_DIR, "StopMotion")
IMAGES_DIR = os.path.join(BASE_DIR, "Imagenes")
SALIDA_DIR = os.path.join(BASE_DIR, "Salida")

TEMP_ROOT = os.path.join(
    tempfile.gettempdir(),
    "POV_CREATOR"
)

for carpeta in [
    PROGRAMA_DIR,
    ASSETS_DIR,
    VIDEOS_DIR,
    STOPMOTION_DIR,
    IMAGES_DIR,
    SALIDA_DIR,
    TEMP_ROOT
]:
    os.makedirs(carpeta, exist_ok=True)


# ============================================================
# FONDOS
# ============================================================
# Recursos visuales independientes para distinguir cada módulo de la interfaz.
# El cambio de fondo funciona únicamente como capa de presentación y no altera
# parámetros de conversión, geometría POV ni archivos generados.

MAIN_BG = os.path.join(ASSETS_DIR, "Main.png")
VIDEO_BG = os.path.join(ASSETS_DIR, "Video.png")
IMAGE_BG = os.path.join(ASSETS_DIR, "Image.png")
IMGSH_BG = os.path.join(ASSETS_DIR, "imgsh.png")
CONFIG_BG = os.path.join(ASSETS_DIR, "Config.png")

FONDOS = {
    "Main": MAIN_BG,
    "Video": VIDEO_BG,
    "Image": IMAGE_BG,
    "imgsh": IMGSH_BG,
    "Config": CONFIG_BG
}

print()
print("==============================================")
print(" VALIDANDO ASSETS")
print("==============================================")
print()

for nombre, ruta in FONDOS.items():
    if not os.path.isfile(ruta):
        raise Exception(
            f"No se encontró el fondo '{nombre}'.\n\n"
            f"Ruta esperada:\n{ruta}"
        )
    print(f"✅ {nombre}")


# ============================================================
# COMPROBAR FFMPEG
# ============================================================
# Validación previa de la herramienta encargada de codificación de video.
# FFmpeg permite normalizar resolución, FPS, códec, formato de píxel y audio.
# H.264 + YUV420P favorece compatibilidad con navegadores, Drive y mensajería.

try:
    subprocess.run(
        ["ffmpeg", "-version"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )
except Exception:
    raise Exception("FFmpeg no está disponible en Colab.")


# ============================================================
# FONDOS BASE64
# ============================================================
# Incrustar los fondos directamente dentro del CSS de la aplicación.
# Evitar dependencias de rutas locales al abrir Gradio mediante enlace público.
# Mantener intactos los archivos originales almacenados en Google Drive.

def imagen_a_base64(ruta):
    mime, _ = mimetypes.guess_type(ruta)
    if mime is None:
        mime = "image/png"

    with open(ruta, "rb") as archivo:
        contenido = base64.b64encode(
            archivo.read()
        ).decode("utf-8")

    return f"data:{mime};base64,{contenido}"


MAIN_DATA = imagen_a_base64(MAIN_BG)
VIDEO_DATA = imagen_a_base64(VIDEO_BG)
IMAGE_DATA = imagen_a_base64(IMAGE_BG)
IMGSH_DATA = imagen_a_base64(IMGSH_BG)
CONFIG_DATA = imagen_a_base64(CONFIG_BG)


# ============================================================
# UTILIDADES
# ============================================================
# Funciones auxiliares para nombres, rutas, selección de archivos y destinos.
# Centralizar validaciones antes de entrar a los algoritmos de conversión.
# Reducir errores por caracteres inválidos, rutas inexistentes o archivos fuera
# de la carpeta de trabajo definida para TESIS_POV.

def limpiar_nombre(nombre):
    if nombre is None:
        nombre = ""

    nombre = nombre.strip()

    if not nombre:
        raise ValueError("Debes escribir un nombre para el archivo.")

    nombre = re.sub(r'[\\/:*?"<>|]', "_", nombre)
    nombre = nombre.strip(" .")

    if not nombre:
        raise ValueError("El nombre del archivo no es válido.")

    return nombre[:100]


def orden_natural(texto):
    texto = os.path.basename(texto)

    return [
        int(parte) if parte.isdigit() else parte.lower()
        for parte in re.split(r"(\d+)", texto)
    ]


def obtener_lista_rutas(valor):
    if valor is None:
        return []

    if not isinstance(valor, list):
        valor = [valor]

    rutas = []

    for elemento in valor:
        if elemento is None:
            continue

        if isinstance(elemento, str):
            ruta = elemento
        elif hasattr(elemento, "path"):
            ruta = elemento.path
        elif hasattr(elemento, "name"):
            ruta = elemento.name
        else:
            continue

        rutas.append(str(ruta))

    return rutas


def normalizar_ruta_drive(ruta):
    if not ruta:
        raise ValueError("Ruta de Drive vacía.")

    if os.path.isabs(ruta):
        ruta_absoluta = os.path.realpath(ruta)
    else:
        ruta_absoluta = os.path.realpath(
            os.path.join(BASE_DIR, ruta)
        )

    base_real = os.path.realpath(BASE_DIR)

    try:
        pertenece = (
            os.path.commonpath(
                [ruta_absoluta, base_real]
            )
            == base_real
        )
    except Exception:
        pertenece = False

    if not pertenece:
        raise ValueError(
            "El archivo seleccionado está fuera de TESIS_POV."
        )

    if not os.path.isfile(ruta_absoluta):
        raise ValueError(
            f"No se encontró el archivo:\n{ruta_absoluta}"
        )

    return ruta_absoluta


def resolver_entrada(
    origen,
    archivos_dispositivo,
    archivos_drive
):
    if origen == "Dispositivo":
        rutas = obtener_lista_rutas(
            archivos_dispositivo
        )

    elif origen == "Google Drive":
        rutas_originales = obtener_lista_rutas(
            archivos_drive
        )

        rutas = [
            normalizar_ruta_drive(ruta)
            for ruta in rutas_originales
        ]

    else:
        raise ValueError(
            "Origen de archivos no válido."
        )

    rutas = [
        ruta
        for ruta in rutas
        if ruta and os.path.isfile(ruta)
    ]

    if not rutas:
        raise ValueError(
            "No seleccionaste ningún archivo."
        )

    return rutas


def crear_trabajo_temporal(prefijo):
    return tempfile.mkdtemp(
        prefix=prefijo + "_",
        dir=TEMP_ROOT
    )


def mostrar_origen(origen):
    dispositivo = origen == "Dispositivo"

    return (
        gr.update(visible=dispositivo),
        gr.update(visible=not dispositivo)
    )


def destino_flags(destino):
    guardar_drive = destino in [
        "Google Drive",
        "Drive + descarga"
    ]

    permitir_descarga = destino in [
        "Descargar al dispositivo",
        "Drive + descarga"
    ]

    return guardar_drive, permitir_descarga


# ============================================================
# CONVERSION DE IMAGEN 200x200
# Lógica compartida por Crear Animación y Adaptar Imagen.
# ============================================================
# Normalización espacial antes de cualquier conversión POV.
# RECORTAR: conservar proporción y llenar 200 x 200 eliminando excedentes.
# AJUSTAR: conservar todo el contenido y completar espacios con negro.
# ESTIRAR: ocupar 200 x 200 sin conservar necesariamente la proporción.
# La conversión final a RGB elimina ambigüedades de alfa o modos de color.

def convertir_imagen_pov(ruta, modo):
    imagen = Image.open(ruta)
    formato_original = imagen.format
    tamano_original = imagen.size
    modo_original = imagen.mode

    imagen = ImageOps.exif_transpose(imagen)

    if imagen.mode in ("RGBA", "LA", "P"):
        imagen_rgba = imagen.convert("RGBA")

        fondo = Image.new(
            "RGB",
            imagen_rgba.size,
            COLOR_FONDO
        )

        fondo.paste(
            imagen_rgba,
            mask=imagen_rgba.getchannel("A")
        )

        imagen = fondo
    else:
        imagen = imagen.convert("RGB")

    if modo == "RECORTAR":
        imagen_final = ImageOps.fit(
            imagen,
            (ANCHO, ALTO),
            method=Image.Resampling.LANCZOS,
            centering=(0.5, 0.5)
        )

    elif modo == "AJUSTAR":
        imagen_redimensionada = ImageOps.contain(
            imagen,
            (ANCHO, ALTO),
            method=Image.Resampling.LANCZOS
        )

        imagen_final = Image.new(
            "RGB",
            (ANCHO, ALTO),
            COLOR_FONDO
        )

        x = (
            ANCHO -
            imagen_redimensionada.width
        ) // 2

        y = (
            ALTO -
            imagen_redimensionada.height
        ) // 2

        imagen_final.paste(
            imagen_redimensionada,
            (x, y)
        )

    elif modo == "ESTIRAR":
        imagen_final = imagen.resize(
            (ANCHO, ALTO),
            Image.Resampling.LANCZOS
        )

    else:
        raise ValueError(
            "Modo inválido. Usa RECORTAR, AJUSTAR o ESTIRAR."
        )

    imagen_final = imagen_final.convert("RGB")

    return (
        imagen_final,
        formato_original,
        tamano_original,
        modo_original
    )


# ============================================================
# FFPROBE
# ============================================================
# Lectura técnica del video antes y después de la conversión.
# Verificar resolución, duración, FPS, códec, formato de píxel y número de frames.
# Utilizar estos datos como control de calidad sin modificar el contenido visual.

def analizar_video(ruta, contar_frames=True):
    comando = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0"
    ]

    if contar_frames:
        comando.append("-count_frames")

    comando.extend(
        [
            "-show_entries",
            (
                "stream="
                "codec_name,"
                "width,"
                "height,"
                "r_frame_rate,"
                "pix_fmt,"
                "nb_read_frames:"
                "format=duration"
            ),
            "-of", "json",
            ruta
        ]
    )

    resultado = subprocess.run(
        comando,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if resultado.returncode != 0:
        raise ValueError(
            "FFprobe no pudo analizar el video."
        )

    datos = json.loads(resultado.stdout)
    streams = datos.get("streams", [])

    if not streams:
        raise ValueError(
            "El archivo no contiene una pista de video válida."
        )

    stream = streams[0]

    duracion = datos.get(
        "format",
        {}
    ).get(
        "duration",
        "0"
    )

    try:
        duracion = float(duracion)
    except Exception:
        duracion = 0.0

    return {
        "codec": stream.get(
            "codec_name",
            "desconocido"
        ),
        "width": int(
            stream.get(
                "width",
                0
            ) or 0
        ),
        "height": int(
            stream.get(
                "height",
                0
            ) or 0
        ),
        "fps": stream.get(
            "r_frame_rate",
            "desconocido"
        ),
        "pix_fmt": stream.get(
            "pix_fmt",
            "desconocido"
        ),
        "frames": stream.get(
            "nb_read_frames",
            "desconocido"
        ),
        "duration": duracion
    }


# ============================================================
# CREAR ANIMACIÓN
# ============================================================
# Construcción de una animación a partir de 1 a 69 imágenes estáticas.
# Distribución temporal automática hasta completar exactamente 69 frames.
# 69 imágenes: una aparición por imagen.
# Menos de 69: repetición equilibrada para ocupar toda la secuencia.
# Salida normalizada: MP4, 200 x 200, 10 FPS, H.264, YUV420P y sin audio.
# La carpeta interna StopMotion permanece por compatibilidad con la estructura
# de almacenamiento ya utilizada durante el desarrollo.

def guardar_stopmotion_en_drive(
    nombre,
    carpeta_frames_temporal,
    video_temporal_final
):
    proyecto_drive = os.path.join(
        STOPMOTION_DIR,
        nombre
    )

    frames_drive = os.path.join(
        proyecto_drive,
        "Imagenes_200x200"
    )

    video_drive = os.path.join(
        VIDEOS_DIR,
        nombre + ".mp4"
    )

    if os.path.exists(proyecto_drive):
        shutil.rmtree(proyecto_drive)

    os.makedirs(
        proyecto_drive,
        exist_ok=True
    )

    shutil.copytree(
        carpeta_frames_temporal,
        frames_drive
    )

    shutil.copy2(
        video_temporal_final,
        video_drive
    )

    return (
        proyecto_drive,
        frames_drive,
        video_drive
    )


def generar_stop_motion(
    origen,
    archivos_dispositivo,
    archivos_drive,
    destino,
    modo,
    nombre_video,
    progress=gr.Progress()
):
    try:
        nombre_video = limpiar_nombre(
            nombre_video
        )

        rutas = resolver_entrada(
            origen,
            archivos_dispositivo,
            archivos_drive
        )

        rutas = sorted(
            rutas,
            key=orden_natural
        )

        cantidad_imagenes = len(rutas)

        if cantidad_imagenes < 1:
            raise ValueError(
                "Debes seleccionar al menos una imagen."
            )

        if cantidad_imagenes > TOTAL_FRAMES_FINAL:
            raise ValueError(
                f"Seleccionaste {cantidad_imagenes} imágenes.\n\n"
                f"El máximo permitido es {TOTAL_FRAMES_FINAL}."
            )

        trabajo = crear_trabajo_temporal(
            "stopmotion"
        )

        carpeta_frames = os.path.join(
            trabajo,
            "Imagenes_200x200"
        )

        os.makedirs(
            carpeta_frames,
            exist_ok=True
        )

        imagenes_convertidas = []

        for indice, ruta in enumerate(
            rutas,
            start=1
        ):
            progress(
                (indice / cantidad_imagenes) * 0.35,
                desc=(
                    f"Convirtiendo imagen "
                    f"{indice}/{cantidad_imagenes}"
                )
            )

            try:
                imagen_final, _, _, _ = (
                    convertir_imagen_pov(
                        ruta,
                        modo
                    )
                )
            except Exception as error:
                raise ValueError(
                    "No se pudo procesar:\n"
                    f"{os.path.basename(ruta)}\n\n"
                    f"{error}"
                )

            nombre_bmp = (
                f"imagen_{indice:03d}.bmp"
            )

            ruta_bmp = os.path.join(
                carpeta_frames,
                nombre_bmp
            )

            imagen_final.save(
                ruta_bmp,
                "BMP"
            )

            imagenes_convertidas.append(
                ruta_bmp
            )

        if cantidad_imagenes == TOTAL_FRAMES_FINAL:
            repeticiones = [
                1
                for _ in range(
                    cantidad_imagenes
                )
            ]
        else:
            repeticiones_base = (
                TOTAL_FRAMES_FINAL
                // cantidad_imagenes
            )

            sobrantes = (
                TOTAL_FRAMES_FINAL
                % cantidad_imagenes
            )

            repeticiones = []

            for i in range(
                cantidad_imagenes
            ):
                cantidad = repeticiones_base

                if i < sobrantes:
                    cantidad += 1

                repeticiones.append(
                    cantidad
                )

        if sum(repeticiones) != TOTAL_FRAMES_FINAL:
            raise ValueError(
                "Error calculando los 69 frames."
            )

        video_mp4v = os.path.join(
            trabajo,
            nombre_video + "_TEMP.mp4"
        )

        fourcc = cv2.VideoWriter_fourcc(
            *"mp4v"
        )

        escritor = cv2.VideoWriter(
            video_mp4v,
            fourcc,
            FPS,
            (ANCHO, ALTO)
        )

        if not escritor.isOpened():
            raise ValueError(
                "No se pudo iniciar VideoWriter."
            )

        contador = 0

        for i, ruta_bmp in enumerate(
            imagenes_convertidas
        ):
            imagen_cv = cv2.imread(
                ruta_bmp
            )

            if imagen_cv is None:
                escritor.release()

                raise ValueError(
                    "No se pudo leer:\n"
                    f"{ruta_bmp}"
                )

            if (
                imagen_cv.shape[1] != ANCHO
                or
                imagen_cv.shape[0] != ALTO
            ):
                imagen_cv = cv2.resize(
                    imagen_cv,
                    (ANCHO, ALTO),
                    interpolation=cv2.INTER_LANCZOS4
                )

            for _ in range(
                repeticiones[i]
            ):
                escritor.write(
                    imagen_cv
                )

                contador += 1

                progress(
                    0.35
                    +
                    (
                        contador
                        /
                        TOTAL_FRAMES_FINAL
                    )
                    * 0.35,
                    desc=(
                        f"Creando frame "
                        f"{contador}/"
                        f"{TOTAL_FRAMES_FINAL}"
                    )
                )

        escritor.release()

        if not os.path.isfile(video_mp4v):
            raise ValueError(
                "No se creó el video temporal."
            )

        video_final = os.path.join(
            trabajo,
            nombre_video + ".mp4"
        )

        progress(
            0.75,
            desc="Codificando H.264"
        )

        comando = [
            "ffmpeg",
            "-y",
            "-i", video_mp4v,
            "-an",
            "-c:v", "libx264",
            "-profile:v", "baseline",
            "-level", "3.0",
            "-pix_fmt", "yuv420p",
            "-r", str(FPS),
            "-crf", "18",
            "-preset", "medium",
            "-movflags", "+faststart",
            video_final
        ]

        resultado = subprocess.run(
            comando,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if resultado.returncode != 0:
            raise ValueError(
                "FFmpeg no pudo crear el video H.264.\n\n"
                +
                resultado.stderr[-1200:]
            )

        try:
            os.remove(video_mp4v)
        except Exception:
            pass

        progress(
            0.90,
            desc="Verificando video"
        )

        info = analizar_video(
            video_final
        )

        try:
            frames_finales = int(
                info["frames"]
            )
        except Exception:
            frames_finales = 0

        if frames_finales != 69:
            raise ValueError(
                "El video final no contiene "
                "exactamente 69 frames.\n\n"
                f"Detectados: {frames_finales}"
            )

        if (
            info["width"] != 200
            or
            info["height"] != 200
        ):
            raise ValueError(
                "La resolución final no es 200 x 200."
            )

        if info["codec"] != "h264":
            raise ValueError(
                "El codec final no es H.264."
            )

        if info["pix_fmt"] != "yuv420p":
            raise ValueError(
                "El formato final no es YUV420P."
            )

        guardar_drive, permitir_descarga = (
            destino_flags(destino)
        )

        ruta_drive = ""
        frames_drive = ""

        if guardar_drive:
            progress(
                0.95,
                desc="Guardando en Google Drive"
            )

            (
                _,
                frames_drive,
                ruta_drive
            ) = guardar_stopmotion_en_drive(
                nombre_video,
                carpeta_frames,
                video_final
            )

        tamano_mb = (
            os.path.getsize(video_final)
            /
            (1024 * 1024)
        )

        distribucion = ", ".join(
            str(x)
            for x in repeticiones
        )

        if len(distribucion) > 180:
            distribucion = (
                distribucion[:180]
                + "..."
            )

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Google Drive:**  \n"
                f"`{ruta_drive}`"
                "\n\n"
                "🖼️ **Frames 200×200:**  \n"
                f"`{frames_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar MP4** "
                "para guardar el archivo en tu dispositivo."
            )

        estado = f"""
### ✅ ANIMACIÓN GENERADA

**Imágenes originales:** {cantidad_imagenes}
**Frames finales:** {frames_finales}
**Resolución:** {info["width"]} × {info["height"]}
**FPS:** 10
**Duración:** {info["duration"]:.2f} s
**Codec:** {info["codec"]}
**Pixel format:** {info["pix_fmt"]}
**Tamaño:** {tamano_mb:.2f} MB

**Distribución de frames:**
`{distribucion}`

{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            video_final
            if permitir_descarga
            else None
        )

        return (
            estado,
            video_final,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))


# ============================================================
# ADAPTAR VIDEO
# ============================================================
# Normalización de videos existentes al formato de entrada del sistema POV.
# Conservar aproximadamente la duración original mediante conversión a 10 FPS.
# No limitar aquí a 69 frames: aplicar la selección uniforme más adelante, dentro
# del generador imgs.h de animación, conservando la relación temporal original.

def guardar_video_en_drive(
    nombre,
    video_temporal_final
):
    video_drive = os.path.join(
        VIDEOS_DIR,
        nombre + ".mp4"
    )

    shutil.copy2(
        video_temporal_final,
        video_drive
    )

    return video_drive


def convertir_video_pov(
    origen,
    archivo_dispositivo,
    archivo_drive,
    destino,
    modo,
    nombre_video,
    progress=gr.Progress()
):
    try:
        nombre_video = limpiar_nombre(
            nombre_video
        )

        rutas = resolver_entrada(
            origen,
            archivo_dispositivo,
            archivo_drive
        )

        if len(rutas) != 1:
            raise ValueError(
                "Selecciona solamente un video."
            )

        input_video = rutas[0]

        progress(
            0.10,
            desc="Analizando video original"
        )

        original = analizar_video(
            input_video,
            contar_frames=False
        )

        if modo == "RECORTAR":
            filtro = (
                f"scale="
                f"{ANCHO}:{ALTO}:"
                f"force_original_aspect_ratio=increase,"
                f"crop="
                f"{ANCHO}:{ALTO},"
                f"fps={FPS},"
                f"setsar=1"
            )

        elif modo == "AJUSTAR":
            filtro = (
                f"scale="
                f"{ANCHO}:{ALTO}:"
                f"force_original_aspect_ratio=decrease,"
                f"pad="
                f"{ANCHO}:{ALTO}:"
                f"(ow-iw)/2:"
                f"(oh-ih)/2:"
                f"black,"
                f"fps={FPS},"
                f"setsar=1"
            )

        elif modo == "ESTIRAR":
            filtro = (
                f"scale="
                f"{ANCHO}:{ALTO},"
                f"fps={FPS},"
                f"setsar=1"
            )

        else:
            raise ValueError(
                "Modo de ajuste desconocido."
            )

        trabajo = crear_trabajo_temporal(
            "video"
        )

        output_video = os.path.join(
            trabajo,
            nombre_video + ".mp4"
        )

        progress(
            0.25,
            desc="Convirtiendo video"
        )

        comando = [
            "ffmpeg",
            "-y",
            "-i", input_video,
            "-vf", filtro,
            "-c:v", "libx264",
            "-profile:v", "baseline",
            "-level", "3.0",
            "-pix_fmt", "yuv420p",
            "-crf", "18",
            "-preset", "medium",
            "-an",
            "-movflags", "+faststart",
            output_video
        ]

        resultado = subprocess.run(
            comando,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if resultado.returncode != 0:
            raise ValueError(
                "FFmpeg no pudo convertir el video.\n\n"
                +
                resultado.stderr[-1200:]
            )

        progress(
            0.78,
            desc="Verificando formato"
        )

        final = analizar_video(
            output_video
        )

        if (
            final["width"] != 200
            or
            final["height"] != 200
        ):
            raise ValueError(
                "La resolución final no es 200 x 200."
            )

        if final["codec"] != "h264":
            raise ValueError(
                "El codec final no es H.264."
            )

        if final["pix_fmt"] != "yuv420p":
            raise ValueError(
                "El pixel format final no es YUV420P."
            )

        guardar_drive, permitir_descarga = (
            destino_flags(destino)
        )

        ruta_drive = ""

        if guardar_drive:
            progress(
                0.92,
                desc="Guardando en Google Drive"
            )

            ruta_drive = guardar_video_en_drive(
                nombre_video,
                output_video
            )

        tamano_mb = (
            os.path.getsize(output_video)
            /
            (1024 * 1024)
        )

        try:
            frames_finales = int(
                final["frames"]
            )
        except Exception:
            frames_finales = "desconocido"

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Guardado en Google Drive:**  \n"
                f"`{ruta_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar MP4** "
                "para guardarlo en tu dispositivo."
            )

        estado = f"""
### ✅ VIDEO CONVERTIDO CORRECTAMENTE

#### Original

**Archivo:** `{os.path.basename(input_video)}`
**Resolución:** {original["width"]} × {original["height"]}
**Codec:** {original["codec"]}
**FPS:** {original["fps"]}
**Duración:** {original["duration"]:.2f} s

#### Resultado POV

**Resolución:** {final["width"]} × {final["height"]}
**FPS:** {final["fps"]}
**Frames:** {frames_finales}
**Duración:** {final["duration"]:.2f} s
**Codec:** {final["codec"]}
**Pixel format:** {final["pix_fmt"]}
**Audio:** NO
**Fast Start:** SÍ
**Tamaño:** {tamano_mb:.2f} MB

{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            output_video
            if permitir_descarga
            else None
        )

        return (
            estado,
            output_video,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))


# ============================================================
# ADAPTAR IMAGEN
# ============================================================
# Preparación de una imagen individual para el conversor POV de imagen fija.
# Generar BMP de 200 x 200 píxeles en modo RGB y mantener fondo negro cuando
# exista transparencia o espacio libre. Este BMP actúa como entrada controlada
# para la generación posterior de frame5[][100].

def convertir_imagen_interfaz(
    origen,
    archivo_dispositivo,
    archivo_drive,
    destino,
    modo,
    nombre_salida,
    progress=gr.Progress()
):
    try:
        nombre_salida = limpiar_nombre(
            nombre_salida
        )

        rutas = resolver_entrada(
            origen,
            archivo_dispositivo,
            archivo_drive
        )

        if len(rutas) != 1:
            raise ValueError(
                "Selecciona solamente una imagen."
            )

        input_image = rutas[0]

        progress(
            0.20,
            desc="Abriendo imagen"
        )

        (
            imagen_final,
            formato_original,
            tamano_original,
            modo_original
        ) = convertir_imagen_pov(
            input_image,
            modo
        )

        trabajo = crear_trabajo_temporal(
            "imagen"
        )

        output_bmp = os.path.join(
            trabajo,
            nombre_salida + ".bmp"
        )

        progress(
            0.60,
            desc="Creando BMP 200×200"
        )

        imagen_final.save(
            output_bmp,
            "BMP"
        )

        # Validación final
        verificacion = Image.open(
            output_bmp
        )

        if verificacion.size != (200, 200):
            raise ValueError(
                "La imagen final no quedó en 200×200."
            )

        if verificacion.mode != "RGB":
            raise ValueError(
                "La imagen final no quedó en modo RGB."
            )

        guardar_drive, permitir_descarga = (
            destino_flags(destino)
        )

        ruta_drive = ""

        if guardar_drive:
            progress(
                0.82,
                desc="Guardando en Google Drive"
            )

            ruta_drive = os.path.join(
                IMAGES_DIR,
                nombre_salida + ".bmp"
            )

            shutil.copy2(
                output_bmp,
                ruta_drive
            )

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Guardado en Google Drive:**  \n"
                f"`{ruta_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar BMP** "
                "para guardarlo en tu dispositivo."
            )

        estado = f"""
### ✅ IMAGEN CONVERTIDA CORRECTAMENTE

#### Original

**Archivo:** `{os.path.basename(input_image)}`
**Formato:** {formato_original or "desconocido"}
**Resolución:** {tamano_original[0]} × {tamano_original[1]}
**Modo:** {modo_original}

#### Resultado POV

**Resolución:** 200 × 200
**Modo:** RGB
**Formato:** BMP
**Ajuste utilizado:** {modo}

{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            output_bmp
            if permitir_descarga
            else None
        )

        return (
            estado,
            input_image,
            output_bmp,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))




# ============================================================
# OPTIMIZACION POV
# ============================================================
# Utilidades opcionales para mejorar la legibilidad de contenido sobre
# fondo negro sin alterar la estructura final de imgs.h.
#
# MODO NORMAL:
#   - Conserva el muestreo original de un punto por muestra polar.
#
# MODO OPTIMIZADO POV:
#   - Detecta el área útil del contenido.
#   - Reencuadra para aprovechar mejor el disco POV.
#   - Aplica un realce suave de color.
#   - Usa supersampling 3x3 ponderado para suavizar bordes.
#   - Da mayor peso al contenido respecto al fondo negro.
# ============================================================

BLACK_THRESHOLD = 16
POV_MARGIN_RATIO = 0.08


def es_modo_optimizado(modo_conversion):
    return str(modo_conversion).strip().lower() == "optimizado pov"


def detectar_mascara_contenido(frame_bgr):
    if frame_bgr is None or frame_bgr.size == 0:
        return None

    max_channel = np.max(frame_bgr, axis=2)
    mask = (max_channel > BLACK_THRESHOLD).astype(np.uint8) * 255

    kernel = np.ones((3, 3), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.dilate(mask, kernel, iterations=1)
    return mask


def detectar_bbox_contenido(frame_bgr):
    mask = detectar_mascara_contenido(frame_bgr)
    if mask is None:
        return None

    coords = cv2.findNonZero(mask)
    if coords is None:
        return None

    x, y, w, h = cv2.boundingRect(coords)
    return (x, y, x + w - 1, y + h - 1)


def expandir_bbox(bbox, width, height, margin_ratio=POV_MARGIN_RATIO):
    if bbox is None:
        return (0, 0, width - 1, height - 1)

    x1, y1, x2, y2 = bbox
    bw = max(1, x2 - x1 + 1)
    bh = max(1, y2 - y1 + 1)
    margen_x = max(2, int(round(bw * margin_ratio)))
    margen_y = max(2, int(round(bh * margin_ratio)))

    x1 = max(0, x1 - margen_x)
    y1 = max(0, y1 - margen_y)
    x2 = min(width - 1, x2 + margen_x)
    y2 = min(height - 1, y2 + margen_y)

    return (x1, y1, x2, y2)


def reencuadrar_bgr_a_200(frame_bgr, bbox=None):
    h, w = frame_bgr.shape[:2]
    bbox = expandir_bbox(bbox, w, h)
    x1, y1, x2, y2 = bbox

    crop = frame_bgr[y1:y2 + 1, x1:x2 + 1]
    if crop.size == 0:
        crop = frame_bgr.copy()

    ch, cw = crop.shape[:2]
    scale = min(ANCHO / max(1, cw), ALTO / max(1, ch))
    nw = max(1, int(round(cw * scale)))
    nh = max(1, int(round(ch * scale)))

    interpolation = cv2.INTER_AREA if scale < 1 else cv2.INTER_LANCZOS4
    resized = cv2.resize(crop, (nw, nh), interpolation=interpolation)

    canvas = np.zeros((ALTO, ANCHO, 3), dtype=np.uint8)
    ox = (ANCHO - nw) // 2
    oy = (ALTO - nh) // 2
    canvas[oy:oy + nh, ox:ox + nw] = resized
    return canvas


def realzar_color_suave(frame_bgr):
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[..., 1] = np.clip(hsv[..., 1] * 1.18, 0, 255)
    hsv[..., 2] = np.clip(hsv[..., 2] * 1.05, 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)


def preparar_frame_pov(frame_bgr, modo_conversion, bbox_global=None):
    if not es_modo_optimizado(modo_conversion):
        return frame_bgr

    if bbox_global is None:
        bbox = detectar_bbox_contenido(frame_bgr)
    else:
        bbox = bbox_global

    procesado = reencuadrar_bgr_a_200(frame_bgr, bbox)
    procesado = realzar_color_suave(procesado)
    return procesado


def unir_bbox(bbox_a, bbox_b):
    if bbox_a is None:
        return bbox_b
    if bbox_b is None:
        return bbox_a

    return (
        min(bbox_a[0], bbox_b[0]),
        min(bbox_a[1], bbox_b[1]),
        max(bbox_a[2], bbox_b[2]),
        max(bbox_a[3], bbox_b[3])
    )


def calcular_bbox_global_video(video_path, selected_frames, progress=None):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise Exception("No se pudo abrir el video para analizar contenido")

    bbox_global = None
    video_frame_index = 0
    selected_position = 0
    total = len(selected_frames)

    while cap.isOpened() and selected_position < total:
        ret, frame = cap.read()
        if not ret:
            break

        if video_frame_index == selected_frames[selected_position]:
            bbox = detectar_bbox_contenido(frame)
            bbox_global = unir_bbox(bbox_global, bbox)
            selected_position += 1

            if progress is not None:
                avance = selected_position / max(1, total)
                progress(0.04 + avance * 0.04, desc=(
                    "Analizando contenido "
                    f"{selected_position}/{total}"
                ))

        video_frame_index += 1

    cap.release()
    return bbox_global


def obtener_rgb888_muestreado(frame_bgr, x, y, modo_conversion):
    alto, ancho = frame_bgr.shape[:2]

    x = max(0, min(alto - 1, int(x)))
    y = max(0, min(ancho - 1, int(y)))

    if not es_modo_optimizado(modo_conversion):
        blue8 = int(frame_bgr[x, y, 0])
        green8 = int(frame_bgr[x, y, 1])
        red8 = int(frame_bgr[x, y, 2])
        return red8, green8, blue8

    offsets = [
        (-1, -1, 1.0), (0, -1, 2.0), (1, -1, 1.0),
        (-1,  0, 2.0), (0,  0, 4.0), (1,  0, 2.0),
        (-1,  1, 1.0), (0,  1, 2.0), (1,  1, 1.0)
    ]

    suma_r = suma_g = suma_b = 0.0
    suma_w = 0.0
    hay_fg = False

    muestras = []
    for dx, dy, base_w in offsets:
        xx = max(0, min(alto - 1, x + dx))
        yy = max(0, min(ancho - 1, y + dy))
        b = float(frame_bgr[xx, yy, 0])
        g = float(frame_bgr[xx, yy, 1])
        r = float(frame_bgr[xx, yy, 2])
        fg = max(r, g, b) > BLACK_THRESHOLD
        muestras.append((r, g, b, fg, base_w))
        if fg:
            hay_fg = True

    for r, g, b, fg, base_w in muestras:
        if hay_fg:
            peso = base_w * (4.0 if fg else 0.20)
        else:
            peso = base_w

        suma_r += r * peso
        suma_g += g * peso
        suma_b += b * peso
        suma_w += peso

    if suma_w <= 0:
        return 0, 0, 0

    red8 = int(round(suma_r / suma_w))
    green8 = int(round(suma_g / suma_w))
    blue8 = int(round(suma_b / suma_w))

    red8 = max(0, min(255, red8))
    green8 = max(0, min(255, green8))
    blue8 = max(0, min(255, blue8))

    return red8, green8, blue8

# ============================================================
# imgs.h DESDE ANIMACION
# ============================================================
# Puente entre video preparado y firmware ESP32.
# Seleccionar un máximo de 69 frames distribuidos uniformemente desde el primero
# hasta el último, evitando concentrar la reducción en una sola zona del video.
# Convertir cada muestra RGB888 a RGB565 para reducir memoria y coincidir con el
# formato esperado por la lógica de visualización embebida.
#
# Un frame almacenado ocupa aproximadamente:
# 60 posiciones angulares x 100 posiciones radiales x 2 bytes = 12 000 bytes.
#
# NUCLEO CRITICO:
# Se conserva la estructura del conversor suministrado:
#
# #define POV_MODE_ANIMATION
# #define POV_ORIGINAL_FRAMES ...
# #define POV_STORED_FRAMES ...
# const uint16_t frames[][60][100] PROGMEM = {
#
# También se conserva la geometría y RGB565.
# ============================================================

def crear_imgsh_animacion_sagrado(
    VIDEO_PATH,
    OUTPUT_PATH,
    modo_conversion="Normal",
    progress=None
):
    # ==========================================
    # CONFIGURACION
    # ==========================================

    # Límite de almacenamiento compatible con la partición OTA empleada.
    # Evitar aumentar este valor sin revisar nuevamente el tamaño compilado.
    MAX_FRAMES = 69

    # Paso angular del sistema POV: 360 / 6 = 60 muestras por revolución.
    resolution = 6

    NUM_LEDS = 200
    HALF_LEDS = int(NUM_LEDS / 2)

    ROWS = int(360 / resolution)


    # ==========================================
    # ABRIR VIDEO
    # ==========================================

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        raise Exception("No se pudo abrir el video")


    # ==========================================
    # INFORMACION DEL VIDEO
    # ==========================================

    TOTAL_FRAMES = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    FPS_VIDEO = cap.get(
        cv2.CAP_PROP_FPS
    )

    WIDTH = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    HEIGHT = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )


    # ==========================================
    # CALCULAR FRAMES A CONSERVAR
    # ==========================================

    if TOTAL_FRAMES <= MAX_FRAMES:

        STORED_FRAMES = TOTAL_FRAMES

        selected_frames = list(
            range(TOTAL_FRAMES)
        )

    else:

        STORED_FRAMES = MAX_FRAMES

        selected_frames = []

        for i in range(STORED_FRAMES):

            index = round(
                i *
                (TOTAL_FRAMES - 1) /
                (STORED_FRAMES - 1)
            )

            selected_frames.append(index)

    cap.release()

    bbox_global = None
    if es_modo_optimizado(modo_conversion):
        bbox_global = calcular_bbox_global_video(
            VIDEO_PATH,
            selected_frames,
            progress=progress
        )

    # Reabrir video para conversión real
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        raise Exception("No se pudo reabrir el video")

    # ==========================================
    # CREAR ARCHIVO imgs.h
    # ==========================================

    f = open(
        OUTPUT_PATH,
        'w'
    )
    # ==========================================
    # CABECERA
    # ==========================================
    f.write(
        "//=====================================\n"
        "// Archivo generado automaticamente\n"
        "// Conversor POV Animacion\n"
        "//=====================================\n\n"
        "#pragma once\n"
        "#include <Arduino.h>\n"
        "#include <pgmspace.h>\n\n"
        "#define POV_MODE_ANIMATION\n\n"
        f"#define POV_ORIGINAL_FRAMES {TOTAL_FRAMES}\n"
        f"#define POV_STORED_FRAMES {STORED_FRAMES}\n\n"
        "const uint16_t frames[][60][100] PROGMEM = {\n"
    )


    # ==========================================
    # CONVERTIR FRAMES
    # ==========================================

    stored_cont = 0

    video_frame_index = 0

    selected_position = 0


    while cap.isOpened():

        ret, frame = cap.read()


        if not ret:

            break


        # ======================================
        # ¿ESTE FRAME SE GUARDA?
        # ======================================

        if (
            selected_position < len(selected_frames)
            and
            video_frame_index ==
            selected_frames[selected_position]
        ):

            frame = preparar_frame_pov(
                frame,
                modo_conversion,
                bbox_global=bbox_global
            )

            if progress is not None:
                avance = (
                    selected_position
                    /
                    max(
                        1,
                        len(selected_frames)
                    )
                )

                progress(
                    0.10 + avance * 0.80,
                    desc=(
                        "Procesando frame "
                        f"{selected_position + 1}/"
                        f"{len(selected_frames)}"
                    )
                )

            degrees = 0
            row = 0

            img_converted = [
                [0 for k in range(HALF_LEDS)]
                for l in range(ROWS)
            ]

            # ==================================
            # CONVERSION POLAR POV
            # ==================================

            while degrees < 360:

                for j in range(0, HALF_LEDS):

                    if degrees <= 90:
                        degrees_aux = 90 - degrees

                        x = int(
                            HALF_LEDS +
                            (HALF_LEDS - j)
                            *
                            math.sin(
                                math.radians(degrees_aux)
                            )
                        )

                        y = int(
                            HALF_LEDS -
                            (HALF_LEDS - j)
                            *
                            math.cos(
                                math.radians(degrees_aux)
                            )
                        )

                    elif degrees <= 180:
                        degrees_aux = degrees - 90

                        x = int(
                            HALF_LEDS -
                            (HALF_LEDS - j)
                            *
                            math.sin(
                                math.radians(degrees_aux)
                            )
                        )

                        y = int(
                            HALF_LEDS -
                            (HALF_LEDS - j)
                            *
                            math.cos(
                                math.radians(degrees_aux)
                            )
                        )

                    elif degrees <= 270:
                        degrees_aux = 270 - degrees

                        x = int(
                            HALF_LEDS -
                            (HALF_LEDS - j)
                            *
                            math.sin(
                                math.radians(degrees_aux)
                            )
                        )

                        y = int(
                            HALF_LEDS +
                            (HALF_LEDS - j)
                            *
                            math.cos(
                                math.radians(degrees_aux)
                            )
                        )

                    else:
                        degrees_aux = degrees - 270

                        x = int(
                            HALF_LEDS +
                            (HALF_LEDS - j)
                            *
                            math.sin(
                                math.radians(degrees_aux)
                            )
                        )

                        y = int(
                            HALF_LEDS +
                            (HALF_LEDS - j)
                            *
                            math.cos(
                                math.radians(degrees_aux)
                            )
                        )

                    if x > NUM_LEDS - 1:
                        x = NUM_LEDS - 1

                    if y > NUM_LEDS - 1:
                        y = NUM_LEDS - 1

                    red8, green8, blue8 = obtener_rgb888_muestreado(
                        frame,
                        x,
                        y,
                        modo_conversion
                    )

                    red5 = (red8 * 249 + 1014) >> 11
                    green6 = (green8 * 253 + 505) >> 10
                    blue5 = (blue8 * 249 + 1014) >> 11

                    img_converted[row][j] = (
                        (red5 << 11) +
                        (green6 << 5) +
                        blue5
                    )

                degrees += resolution
                row += 1

            # ==================================
            # ESCRIBIR FRAME
            # ==================================
            if stored_cont != 0:
                f.write(",\n")

            f.write("{\n")

            for i in range(len(img_converted)):

                f.write("{")

                for j in range(len(img_converted[0])):

                    value = img_converted[i][j]
                    hex_value = "0x{:04X}".format(value)
                    f.write(hex_value)

                    if j < len(img_converted[0]) - 1:
                        f.write(",")

                f.write("}")

                if i < len(img_converted) - 1:
                    f.write(",")

                f.write("\n")

            f.write("}")

            stored_cont += 1
            selected_position += 1

        video_frame_index += 1

    cap.release()

    # ==========================================
    # CERRAR ARRAY
    # ==========================================

    f.write("\n};\n")
    f.close()

    removed_frames = TOTAL_FRAMES - STORED_FRAMES
    saved_bytes = max(0, removed_frames) * 12000

    return {
        "width": WIDTH,
        "height": HEIGHT,
        "fps": FPS_VIDEO,
        "total_frames": TOTAL_FRAMES,
        "stored_frames": STORED_FRAMES,
        "removed_frames": removed_frames,
        "saved_bytes": saved_bytes,
        "modo_conversion": modo_conversion
    }


def generar_imgsh_animacion(
    origen,
    archivo_dispositivo,
    archivo_drive,
    destino,
    modo_conversion,
    progress=gr.Progress()
):
    try:
        rutas = resolver_entrada(
            origen,
            archivo_dispositivo,
            archivo_drive
        )

        if len(rutas) != 1:
            raise ValueError(
                "Selecciona solamente un video."
            )

        video_path = rutas[0]

        progress(
            0.03,
            desc="Preparando conversión"
        )

        trabajo = crear_trabajo_temporal(
            "imgsh_animacion"
        )

        output_temp = os.path.join(
            trabajo,
            "imgs.h"
        )

        datos = crear_imgsh_animacion_sagrado(
            video_path,
            output_temp,
            modo_conversion=modo_conversion,
            progress=progress
        )

        if not os.path.isfile(output_temp):
            raise ValueError(
                "No se generó imgs.h."
            )

        guardar_drive, permitir_descarga = (
            destino_flags(destino)
        )

        ruta_drive = ""

        if guardar_drive:
            progress(
                0.94,
                desc="Guardando imgs.h en Drive"
            )

            ruta_drive = os.path.join(
                SALIDA_DIR,
                "imgs.h"
            )

            shutil.copy2(
                output_temp,
                ruta_drive
            )

        advertencia = ""

        if (
            datos["width"] != 200
            or
            datos["height"] != 200
        ):
            advertencia = (
                "\n\n"
                "⚠️ **Advertencia:** el video no es 200×200. "
                "El conversor original espera videos 200×200. "
                "Para evitar problemas usa primero **Adaptar video**."
            )

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Guardado en Google Drive:**  \n"
                f"`{ruta_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar imgs.h** "
                "para guardarlo en tu dispositivo."
            )

        estado = f"""
### ✅ imgs.h DE ANIMACIÓN GENERADO

**Video:** `{os.path.basename(video_path)}`
**Resolución detectada:** {datos["width"]} × {datos["height"]}
**FPS detectado:** {datos["fps"]:.2f}
**Frames originales:** {datos["total_frames"]}
**Frames almacenados:** {datos["stored_frames"]}
**Frames eliminados:** {datos["removed_frames"]}
**Ahorro aproximado:** {datos["saved_bytes"]:,} bytes
**Modo POV:** {datos["modo_conversion"]}

El archivo conserva:

`#define POV_MODE_ANIMATION`
`#define POV_ORIGINAL_FRAMES {datos["total_frames"]}`
`#define POV_STORED_FRAMES {datos["stored_frames"]}`
`const uint16_t frames[][60][100] PROGMEM = {{ ... }};`

{advertencia}

💡 **Modo Optimizado POV** aplica reencuadre del contenido, prioridad al primer plano sobre fondo negro y antialiasing 3×3 ponderado.

{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            output_temp
            if permitir_descarga
            else None
        )

        return (
            estado,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))


# ============================================================
# imgs.h DESDE IMAGEN
# ============================================================
# Conversión de un BMP normalizado a la matriz polar utilizada para imagen fija.
# Recorrer 360 grados en incrementos de 6 grados para obtener 60 filas.
# Muestrear 100 posiciones radiales por cada ángulo y convertir RGB888 a RGB565.
# Mantener la organización frame5[][100] por compatibilidad directa con firmware.
#
# NUCLEO CRITICO:
# Se conserva la estructura del conversor suministrado para
# imagen fija, especialmente:
#
# const uint16_t frame5[][100] PROGMEM = {
# ============================================================

def crear_imgsh_imagen_sagrado(
    IMAGE_PATH,
    OUTPUT_PATH,
    modo_conversion="Normal",
    progress=None
):
    # ==========================================
    # CONFIGURACIÓN
    # ==========================================

    frame = cv2.imread(IMAGE_PATH)

    if frame is None:
        raise Exception("No se pudo leer la imagen")

    alto, ancho = frame.shape[:2]

    if ancho != 200 or alto != 200:
        raise Exception(
            "La imagen no es 200x200. Usa primero ADAPTAR IMAGEN."
        )

    frame = preparar_frame_pov(
        frame,
        modo_conversion,
        bbox_global=None
    )

    # ==========================================
    # CONVERSIÓN POV
    # ==========================================

    degrees = 0
    degrees_aux = 0

    NUM_LEDS = 200
    HALF_LEDS = int(NUM_LEDS / 2)

    row = 0

    # Paso angular fijo del conversor de imagen: 60 filas por revolución.
    resolution = 6

    img_converted = [
        [0 for k in range(HALF_LEDS)]
        for l in range(int(360 / resolution))
    ]

    total_rows = int(360 / resolution)

    while degrees < 360:

        if progress is not None:
            avance = row / max(1, total_rows)
            progress(
                0.10 + avance * 0.80,
                desc=(
                    "Procesando ángulo "
                    f"{row + 1}/{total_rows}"
                )
            )

        for j in range(0, HALF_LEDS):

            if degrees <= 90:

                degrees_aux = 90 - degrees

                x = int(
                    HALF_LEDS +
                    (HALF_LEDS - j) *
                    math.sin(math.radians(degrees_aux))
                )

                y = int(
                    HALF_LEDS -
                    (HALF_LEDS - j) *
                    math.cos(math.radians(degrees_aux))
                )

            elif degrees <= 180:

                degrees_aux = degrees - 90

                x = int(
                    HALF_LEDS -
                    (HALF_LEDS - j) *
                    math.sin(math.radians(degrees_aux))
                )

                y = int(
                    HALF_LEDS -
                    (HALF_LEDS - j) *
                    math.cos(math.radians(degrees_aux))
                )

            elif degrees <= 270:

                degrees_aux = 270 - degrees

                x = int(
                    HALF_LEDS -
                    (HALF_LEDS - j) *
                    math.sin(math.radians(degrees_aux))
                )

                y = int(
                    HALF_LEDS +
                    (HALF_LEDS - j) *
                    math.cos(math.radians(degrees_aux))
                )

            else:

                degrees_aux = degrees - 270

                x = int(
                    HALF_LEDS +
                    (HALF_LEDS - j) *
                    math.sin(math.radians(degrees_aux))
                )

                y = int(
                    HALF_LEDS +
                    (HALF_LEDS - j) *
                    math.cos(math.radians(degrees_aux))
                )

            if x > NUM_LEDS - 1:
                x = NUM_LEDS - 1

            if y > NUM_LEDS - 1:
                y = NUM_LEDS - 1

            red8, green8, blue8 = obtener_rgb888_muestreado(
                frame,
                x,
                y,
                modo_conversion
            )

            red5 = (red8 * 249 + 1014) >> 11
            green6 = (green8 * 253 + 505) >> 10
            blue5 = (blue8 * 249 + 1014) >> 11

            img_converted[row][j] = (
                (red5 << 11) +
                (green6 << 5) +
                blue5
            )

        degrees += resolution
        row += 1

    # ==========================================
    # CREAR imgs.h
    # ==========================================

    f = open(OUTPUT_PATH, 'w', encoding='utf-8')

    f.write(
"""//=====================================
// Archivo generado automáticamente
// Conversor POV Imagen
//=====================================

#pragma once
#include <Arduino.h>

const uint16_t frame5[][100] PROGMEM = {

"""
    )

    for i in range(len(img_converted)):

        f.write("{")

        for j in range(len(img_converted[0])):

            value = img_converted[i][j]

            if value == 0:
                f.write("0x0000")

            elif value <= 15:
                h = hex(value)
                f.write(
                    h[:2] + "000" + h[2:]
                )

            elif value <= 255:
                h = hex(value)
                f.write(
                    h[:2] + "00" + h[2:]
                )

            elif value <= 4095:
                h = hex(value)
                f.write(
                    h[:2] + "0" + h[2:]
                )

            else:
                f.write(hex(value))

            f.write(",")

        f.write("},\n")

    f.write("};")
    f.close()

    return {
        "width": ancho,
        "height": alto,
        "rows": len(img_converted),
        "cols": len(img_converted[0]),
        "modo_conversion": modo_conversion
    }


def generar_imgsh_imagen(
    origen,
    archivo_dispositivo,
    archivo_drive,
    destino,
    modo_conversion,
    progress=gr.Progress()
):
    try:
        rutas = resolver_entrada(
            origen,
            archivo_dispositivo,
            archivo_drive
        )

        if len(rutas) != 1:
            raise ValueError(
                "Selecciona solamente una imagen."
            )

        image_path = rutas[0]

        progress(
            0.03,
            desc="Preparando conversión"
        )

        trabajo = crear_trabajo_temporal(
            "imgsh_imagen"
        )

        output_temp = os.path.join(
            trabajo,
            "imgs.h"
        )

        datos = crear_imgsh_imagen_sagrado(
            image_path,
            output_temp,
            modo_conversion=modo_conversion,
            progress=progress
        )

        if not os.path.isfile(output_temp):
            raise ValueError(
                "No se generó imgs.h."
            )

        guardar_drive, permitir_descarga = destino_flags(destino)

        ruta_drive = ""

        if guardar_drive:
            progress(
                0.94,
                desc="Guardando imgs.h en Drive"
            )

            ruta_drive = os.path.join(
                SALIDA_DIR,
                "imgs.h"
            )

            shutil.copy2(
                output_temp,
                ruta_drive
            )

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Guardado en Google Drive:**  \n"
                f"`{ruta_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar imgs.h** "
                "para guardarlo en tu dispositivo."
            )

        estado = f"""
### ✅ imgs.h DE IMAGEN GENERADO

**Imagen:** `{os.path.basename(image_path)}`
**Resolución detectada:** {datos['width']} × {datos['height']}
**Matriz generada:** {datos['rows']} filas × {datos['cols']} columnas
**Modo POV:** {datos['modo_conversion']}

El archivo conserva la estructura crítica:

`const uint16_t frame5[][100] PROGMEM = {{ ... }};`

⚠️ Esta función espera una imagen **200×200**. Si tu archivo no tiene ese formato, usa primero **Adaptar imagen**.

💡 **Modo Optimizado POV** está recomendado para imágenes con fondo negro y aplica reencuadre del contenido, prioridad al primer plano y antialiasing 3×3 ponderado.

{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            output_temp
            if permitir_descarga
            else None
        )

        return (
            estado,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))


# ============================================================
# CSS
# ============================================================
# Capa exclusivamente visual de POV Creator.
# Definir fondos, tipografías, adaptación a pantalla completa, menús y paneles.
# Mantener esta sección desacoplada de los algoritmos de conversión para permitir
# cambios estéticos sin afectar resultados técnicos ni estructuras imgs.h.

CSS = f"""
@import url('https://fonts.googleapis.com/css2?family=Michroma&family=Orbitron:wght@600;700;800;900&display=swap');

html,
body {{
    margin: 0 !important;
    padding: 0 !important;
    width: 100% !important;
    height: 100% !important;
    overflow: hidden !important;
    background: transparent !important;
}}

.gradio-container {{
    position: fixed !important;
    top: 0 !important;
    left: 0 !important;
    width: 100vw !important;
    height: 100vh !important;
    max-width: none !important;
    margin: 0 !important;
    padding: 0 !important;
    overflow: hidden !important;
    background: transparent !important;
    font-family: 'Orbitron', sans-serif !important;
}}

.gradio-container > .main,
.gradio-container main,
.contain,
.wrap {{
    margin: 0 !important;
    padding: 0 !important;
    max-width: none !important;
    background: transparent !important;
}}

.screen {{
    position: fixed !important;
    top: 0 !important;
    left: 0 !important;
    width: 100vw !important;
    height: 100vh !important;
    margin: 0 !important;
    padding: 0 !important;
    box-sizing: border-box !important;
    overflow: hidden !important;
    background-size: cover !important;
    background-position: center center !important;
    background-repeat: no-repeat !important;
}}

.menu-screen {{
    display: flex !important;
    flex-direction: column !important;
    justify-content: center !important;
    align-items: center !important;
}}

.bg-main {{
    background-image:
        linear-gradient(
            rgba(0,8,30,0.04),
            rgba(0,8,25,0.09)
        ),
        url("{MAIN_DATA}") !important;
}}

.bg-video {{
    background-image:
        linear-gradient(
            rgba(18,0,35,0.03),
            rgba(14,0,30,0.07)
        ),
        url("{VIDEO_DATA}") !important;
}}

.bg-image {{
    background-image:
        linear-gradient(
            rgba(0,20,4,0.03),
            rgba(0,15,3,0.07)
        ),
        url("{IMAGE_DATA}") !important;
}}

.bg-imgsh {{
    background-image:
        linear-gradient(
            rgba(35,0,0,0.03),
            rgba(22,0,0,0.07)
        ),
        url("{IMGSH_DATA}") !important;
}}

.bg-config {{
    background-image:
        linear-gradient(
            rgba(0,0,0,0.03),
            rgba(0,0,0,0.07)
        ),
        url("{CONFIG_DATA}") !important;
}}

.menu-container {{
    width: min(860px, 92vw) !important;
    margin: auto !important;
    padding: 20px !important;
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    gap: 14px !important;
}}

/* Fuente anterior del título */
.pov-title {{
    font-family: 'Michroma', sans-serif !important;
    text-align: center;
    text-transform: uppercase;
    color: #FFFFFF;
    line-height: 1.08;
    margin: 0 0 46px 0;
    letter-spacing: 0.10em;
    -webkit-text-stroke: 1.1px rgba(70,155,255,0.95);
    text-shadow:
        0 0 4px white,
        0 0 10px rgba(110,195,255,1),
        0 0 20px rgba(0,105,255,0.82);
}}

.pov-title .top-line {{
    display: block;
    font-size: clamp(42px, 7vw, 78px);
    margin-bottom: 10px;
}}

.pov-title .bottom-line {{
    display: block;
    font-size: clamp(32px, 5vw, 57px);
}}

.section-title {{
    font-family: 'Orbitron', sans-serif !important;
    font-weight: 800;
    text-align: center;
    color: white;
    font-size: clamp(27px, 4vw, 45px);
    letter-spacing: 0.07em;
    margin: 0 0 28px 0;
    -webkit-text-stroke: 0.8px rgba(65,150,255,0.95);
    text-shadow:
        0 0 3px white,
        0 0 9px #55AFFF,
        0 0 16px #006BFF;
}}

.game-menu {{
    width: min(700px, 88vw) !important;
    margin: 5px auto !important;
}}

.game-menu button {{
    width: 100% !important;
    min-height: 58px !important;
    font-family: 'Orbitron', sans-serif !important;
    font-weight: 800 !important;
    color: white !important;
    font-size: clamp(15px, 2vw, 23px) !important;
    text-transform: uppercase;
    background: rgba(15,18,28,0.38) !important;
    border: 1px solid rgba(255,255,255,0.30) !important;
    text-shadow:
        0 0 3px white,
        0 0 8px #5CB7FF,
        0 0 15px #006EFF !important;
    transition: all 0.15s ease !important;
}}

.game-menu button:hover {{
    border-color: white !important;
    background: rgba(255,255,255,0.06) !important;
    transform: scale(1.015);
}}

.back-menu {{
    margin-top: 18px !important;
}}

.work-wrapper {{
    width: 100vw !important;
    height: 100vh !important;
    overflow-y: auto !important;
    padding: 35px 18px 60px 18px !important;
    box-sizing: border-box !important;
}}

.work-panel {{
    width: min(980px, 94vw) !important;
    margin: 0 auto !important;
    padding: 25px !important;
    box-sizing: border-box !important;
    background: rgba(5,7,18,0.62) !important;
    border: 1px solid rgba(255,255,255,0.38) !important;
    border-radius: 10px !important;
    backdrop-filter: blur(6px);
    box-shadow: 0 0 25px rgba(0,0,0,0.28) !important;
}}

.work-panel,
.work-panel label,
.work-panel span,
.work-panel p,
.work-panel div,
.work-panel input,
.work-panel textarea,
.work-panel button {{
    font-family: 'Orbitron', sans-serif !important;
}}

.tool-info {{
    color: white !important;
    opacity: 0.94;
}}

.help-box {{
    margin-top: -6px !important;
    padding: 10px 14px !important;
    background: rgba(0,0,0,0.18) !important;
    border-left: 3px solid rgba(160,220,255,0.75) !important;
    border-radius: 5px !important;
}}

.action-btn button {{
    font-family: 'Orbitron', sans-serif !important;
    font-weight: 900 !important;
    color: white !important;
    background: rgba(0,85,170,0.55) !important;
    border: 1px solid rgba(160,220,255,0.80) !important;
    text-shadow:
        0 0 4px white,
        0 0 10px #168FFF !important;
}}

.action-btn button:hover {{
    background: rgba(0,120,220,0.62) !important;
    border-color: white !important;
}}

.result-panel {{
    margin-top: 18px !important;
    padding: 15px !important;
    background: rgba(0,0,0,0.22) !important;
    border: 1px solid rgba(255,255,255,0.18) !important;
    border-radius: 8px !important;
}}

.manual-panel {{
    line-height: 1.65 !important;
}}

footer {{
    display: none !important;
}}

@media (max-width: 700px) {{
    .work-wrapper {{
        padding: 20px 8px 50px 8px !important;
    }}

    .work-panel {{
        width: 96vw !important;
        padding: 15px !important;
    }}

    .menu-container {{
        width: 94vw !important;
    }}

    .game-menu {{
        width: 90vw !important;
    }}
}}
"""


# ============================================================
# NAVEGACION
# ============================================================
# Control de visibilidad entre módulos dentro de una sola aplicación Gradio.
# Mantener cargados los componentes y alternar únicamente la pantalla activa.
# Evitar recargas completas al cambiar entre animaciones, imágenes, imgs.h y manual.

NUM_PANTALLAS = 10


def mostrar_pantalla(indice):
    return [
        gr.update(
            visible=(i == indice)
        )
        for i in range(NUM_PANTALLAS)
    ]


def ir_principal():
    return mostrar_pantalla(0)


def ir_animaciones():
    return mostrar_pantalla(1)


def ir_imagenes():
    return mostrar_pantalla(2)


def ir_imgs():
    return mostrar_pantalla(3)


def ir_manual():
    return mostrar_pantalla(4)


def ir_stopmotion():
    return mostrar_pantalla(5)


def ir_video():
    return mostrar_pantalla(6)


def ir_adaptar_imagen():
    return mostrar_pantalla(7)


def ir_imgs_animacion():
    return mostrar_pantalla(8)


def ir_imgs_imagen():
    return mostrar_pantalla(9)


# ============================================================
# TEXTO DE AYUDA DE AJUSTE
# ============================================================
# Explicación breve junto al selector para reducir errores de interpretación.
# Presentar la consecuencia visual de cada método sin modificar sus parámetros.

AYUDA_AJUSTE = """
**¿Qué hace cada modo?**

- **RECORTAR:** conserva la proporción y llena todo el cuadro 200×200. Puede cortar parte de los bordes.
- **AJUSTAR:** conserva toda la imagen/video dentro de 200×200. Si sobra espacio, agrega fondo negro.
- **ESTIRAR:** fuerza el contenido directamente a 200×200. No recorta, pero puede deformarlo.
"""


# ============================================================
# INTERFAZ
# ============================================================
# Integración de controles de entrada, destino, previsualización y navegación.
# Permitir selección desde dispositivo o Google Drive y salida como descarga,
# almacenamiento persistente en Drive o ambas opciones simultáneamente.
# Mantener la lógica de procesamiento en funciones separadas de la presentación.

with gr.Blocks(
    css=CSS,
    title="POV Creator"
) as app:

    # ========================================================
    # 0 - PRINCIPAL
    # ========================================================

    with gr.Column(
        visible=True,
        elem_classes=[
            "screen",
            "menu-screen",
            "bg-main"
        ]
    ) as panel_principal:

        with gr.Column(
            elem_classes=[
                "menu-container"
            ]
        ):

            gr.HTML(
                """
                <div class="pov-title">
                    <span class="top-line">POV</span>
                    <span class="bottom-line">CREATOR</span>
                </div>
                """
            )

            btn_animaciones = gr.Button(
                "ANIMACIONES",
                elem_classes=["game-menu"]
            )

            btn_imagenes = gr.Button(
                "IMÁGENES",
                elem_classes=["game-menu"]
            )

            btn_imgs = gr.Button(
                "GENERAR imgs.h",
                elem_classes=["game-menu"]
            )

            btn_manual = gr.Button(
                "MANUAL",
                elem_classes=["game-menu"]
            )


    # ========================================================
    # 1 - MENU ANIMACIONES
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "menu-screen",
            "bg-video"
        ]
    ) as panel_animaciones:

        with gr.Column(
            elem_classes=[
                "menu-container"
            ]
        ):

            gr.HTML(
                """
                <div class="section-title">
                    ANIMACIONES
                </div>
                """
            )

            btn_stopmotion = gr.Button(
                "CREAR ANIMACIÓN",
                elem_classes=["game-menu"]
            )

            btn_video = gr.Button(
                "ADAPTAR VIDEO EXISTENTE",
                elem_classes=["game-menu"]
            )

            btn_back_animaciones = gr.Button(
                "VOLVER",
                elem_classes=[
                    "game-menu",
                    "back-menu"
                ]
            )


    # ========================================================
    # 2 - MENU IMAGENES
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "menu-screen",
            "bg-image"
        ]
    ) as panel_imagenes:

        with gr.Column(
            elem_classes=[
                "menu-container"
            ]
        ):

            gr.HTML(
                """
                <div class="section-title">
                    IMÁGENES
                </div>
                """
            )

            btn_adaptar_imagen = gr.Button(
                "ADAPTAR IMAGEN",
                elem_classes=["game-menu"]
            )

            btn_back_imagenes = gr.Button(
                "VOLVER",
                elem_classes=[
                    "game-menu",
                    "back-menu"
                ]
            )


    # ========================================================
    # 3 - MENU imgs.h
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "menu-screen",
            "bg-imgsh"
        ]
    ) as panel_imgs:

        with gr.Column(
            elem_classes=[
                "menu-container"
            ]
        ):

            gr.HTML(
                """
                <div class="section-title">
                    GENERAR imgs.h
                </div>
                """
            )

            btn_imgs_animacion = gr.Button(
                "DESDE ANIMACIÓN",
                elem_classes=["game-menu"]
            )

            btn_imgs_imagen = gr.Button(
                "DESDE IMAGEN",
                elem_classes=["game-menu"]
            )

            btn_back_imgs = gr.Button(
                "VOLVER",
                elem_classes=[
                    "game-menu",
                    "back-menu"
                ]
            )


    # ========================================================
    # 4 - MANUAL
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-config"
        ]
    ) as panel_manual:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel",
                    "manual-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        MANUAL
                    </div>
                    """
                )

                # ------------------------------------------------
                # INFORMACION QUE YA ESTABA EN CONFIGURACION
                # ------------------------------------------------

                gr.Markdown(
                    """
### Parámetros actuales del sistema

**Resolución POV:** 200 × 200 🔒
**Máximo de frames almacenados:** 69 🔒
**Formato interno:** RGB565 🔒
**FPS recomendado de preparación:** 10

---

### ESP32 y OTA

El proyecto utiliza una **ESP32-WROOM-32 de 4 MB** y la tabla de particiones:

**`No FS 4MB (2MB APP x2)`**

Esta configuración reserva aproximadamente **2 MB para cada aplicación** y permite mantener **OTA**. El tamaño máximo de aplicación que hemos venido usando es aproximadamente **2.031.616 bytes**.

Para conservar OTA:

- No usar una tabla **No OTA**.
- No cambiar a **Huge APP** para este flujo.
- Si cambias la tabla de particiones en Arduino IDE, debes hacer **una carga por USB** para escribir la nueva tabla; después puedes volver a cargar por OTA.
- Cada frame RGB565 del POV ocupa aproximadamente **12.000 bytes**: 60 filas × 100 valores × 2 bytes.
- El límite actual de **69 frames** protege el espacio disponible para el firmware y las librerías Wi‑Fi/OTA.
- La versión de animación que hemos probado con 69 frames ocupa aproximadamente **1.919.839 bytes**, alrededor del 94 % de la partición de aplicación. Por eso no conviene aumentar el número de frames sin volver a comprobar el tamaño compilado.

---

### Guía rápida de POV Creator

#### 1. Animaciones → Crear Animación

Usa entre **1 y 69 imágenes**. El sistema las adapta a 200×200 y las distribuye hasta crear exactamente **69 frames**, 10 FPS, H.264 y YUV420P.

Puedes elegir:

**Origen**
- Dispositivo: arrastrar archivos o buscarlos.
- Google Drive: seleccionar archivos dentro de `TESIS_POV`.

**Destino**
- Descargar al dispositivo.
- Guardar en Google Drive.
- Drive + descarga.

En Drive, el MP4 queda en:

`TESIS_POV/Videos/`

y las imágenes intermedias 200×200 quedan en:

`TESIS_POV/StopMotion/<Nombre>/Imagenes_200x200/`

#### 2. Animaciones → Adaptar video existente

Convierte un video normal a:

**MP4 · 200×200 · 10 FPS · H.264 · YUV420P · sin audio**

El video conserva aproximadamente su duración. Si tiene más de 69 frames, esa reducción se hace después al generar `imgs.h`.

#### 3. Imágenes → Adaptar imagen

Convierte JPG, PNG, WEBP, BMP y otros formatos admitidos por Pillow a:

**BMP · 200×200 · RGB**

En Drive queda en:

`TESIS_POV/Imagenes/`

#### 4. Generar imgs.h → Desde animación

Selecciona preferiblemente uno de los MP4 que ya salió de **Crear Animación** o **Adaptar video**.

El conversor conserva como máximo **69 frames** y genera exactamente la estructura que utiliza el firmware de animación:

`const uint16_t frames[][60][100] PROGMEM = { ... };`

Si eliges Drive como destino se guarda en:

`TESIS_POV/Salida/imgs.h`

#### 5. Arduino IDE

1. Sustituye el `imgs.h` del proyecto por el generado.
2. Comprueba la placa ESP32 y la tabla `No FS 4MB (2MB APP x2)`.
3. Compila y verifica que el sketch no supere el espacio disponible.
4. Con la ESP32 y el PC en la misma Wi‑Fi, selecciona `POV-ESP32` y realiza la carga OTA.
5. Si acabas de cambiar la tabla de particiones, realiza primero una carga por USB.

> **Importante:** el generador `imgs.h` de animación es una parte crítica del proyecto. Su geometría, formato RGB565, selección de frames y estructura `frames[][60][100]` no deben modificarse sin validar primero el firmware.
                    """
                )

                btn_back_manual = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 5 - STOP MOTION
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-video"
        ]
    ) as panel_stopmotion:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        CREAR ANIMACIÓN
                    </div>
                    """
                )

                gr.Markdown(
                    """
**Resultado:** MP4 · 200×200 · 10 FPS · 69 frames · H.264 · YUV420P
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    stop_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Dispositivo",
                        label="1. Origen de las imágenes"
                    )

                    stop_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino del resultado"
                    )

                stop_files_device = gr.File(
                    label=(
                        "Arrastra aquí entre 1 y 69 imágenes "
                        "o haz clic para buscarlas"
                    ),
                    file_count="multiple",
                    file_types=[
                        "image",
                        ".bmp",
                        ".webp",
                        ".tif",
                        ".tiff"
                    ],
                    type="filepath",
                    visible=True
                )

                stop_files_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="multiple",
                    label=(
                        "Seleccionar imágenes desde TESIS_POV"
                    ),
                    height=280,
                    visible=False
                )

                stop_mode = gr.Dropdown(
                    choices=[
                        "RECORTAR",
                        "AJUSTAR",
                        "ESTIRAR"
                    ],
                    value="RECORTAR",
                    label="Modo de ajuste 200×200"
                )

                gr.Markdown(
                    AYUDA_AJUSTE,
                    elem_classes=["help-box"]
                )

                stop_name = gr.Textbox(
                    label="Nombre del video",
                    placeholder="Ejemplo: Bienvenida"
                )

                stop_generate = gr.Button(
                    "GENERAR ANIMACIÓN",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    stop_status = gr.Markdown()

                    stop_preview = gr.Video(
                        label="Vista previa del MP4"
                    )

                    stop_download = gr.File(
                        label="Descargar MP4",
                        interactive=False
                    )

                    stop_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_stop = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 6 - ADAPTAR VIDEO
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-video"
        ]
    ) as panel_video:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        ADAPTAR VIDEO
                    </div>
                    """
                )

                gr.Markdown(
                    """
**Resultado:** MP4 · 200×200 · 10 FPS · H.264 · YUV420P · sin audio
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    video_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Dispositivo",
                        label="1. Origen del video"
                    )

                    video_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino del resultado"
                    )

                video_file_device = gr.File(
                    label=(
                        "Arrastra el video aquí "
                        "o haz clic para buscarlo"
                    ),
                    file_count="single",
                    file_types=[
                        "video",
                        ".mp4",
                        ".mov",
                        ".avi",
                        ".mkv",
                        ".webm",
                        ".m4v",
                        ".mpeg",
                        ".mpg",
                        ".wmv",
                        ".flv",
                        ".3gp",
                        ".ts",
                        ".mts",
                        ".m2ts"
                    ],
                    type="filepath",
                    visible=True
                )

                video_file_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="single",
                    label=(
                        "Seleccionar video desde TESIS_POV"
                    ),
                    height=280,
                    visible=False
                )

                video_mode = gr.Dropdown(
                    choices=[
                        "RECORTAR",
                        "AJUSTAR",
                        "ESTIRAR"
                    ],
                    value="RECORTAR",
                    label="Modo de ajuste 200×200"
                )

                gr.Markdown(
                    AYUDA_AJUSTE,
                    elem_classes=["help-box"]
                )

                video_name = gr.Textbox(
                    label="Nombre del video final",
                    placeholder="Ejemplo: MinecraftSteve"
                )

                video_generate = gr.Button(
                    "CONVERTIR VIDEO",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    video_status = gr.Markdown()

                    video_preview = gr.Video(
                        label="Vista previa del MP4"
                    )

                    video_download = gr.File(
                        label="Descargar MP4",
                        interactive=False
                    )

                    video_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_video = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 7 - ADAPTAR IMAGEN
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-image"
        ]
    ) as panel_adaptar_imagen:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        ADAPTAR IMAGEN
                    </div>
                    """
                )

                gr.Markdown(
                    """
**Resultado:** BMP · 200×200 · RGB
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    image_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Dispositivo",
                        label="1. Origen de la imagen"
                    )

                    image_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino del resultado"
                    )

                image_file_device = gr.File(
                    label=(
                        "Arrastra una imagen aquí "
                        "o haz clic para buscarla"
                    ),
                    file_count="single",
                    file_types=[
                        "image",
                        ".bmp",
                        ".webp",
                        ".tif",
                        ".tiff"
                    ],
                    type="filepath",
                    visible=True
                )

                image_file_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="single",
                    label=(
                        "Seleccionar imagen desde TESIS_POV"
                    ),
                    height=280,
                    visible=False
                )

                image_mode = gr.Dropdown(
                    choices=[
                        "RECORTAR",
                        "AJUSTAR",
                        "ESTIRAR"
                    ],
                    value="RECORTAR",
                    label="Modo de ajuste 200×200"
                )

                gr.Markdown(
                    AYUDA_AJUSTE,
                    elem_classes=["help-box"]
                )

                image_name = gr.Textbox(
                    label="Nombre del BMP final",
                    placeholder="Ejemplo: Logo_USC"
                )

                image_generate = gr.Button(
                    "CONVERTIR IMAGEN",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    image_status = gr.Markdown()

                    with gr.Row():

                        image_original_preview = gr.Image(
                            label="Original"
                        )

                        image_result_preview = gr.Image(
                            label="Resultado 200×200 BMP"
                        )

                    image_download = gr.File(
                        label="Descargar BMP",
                        interactive=False
                    )

                    image_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_adaptar_imagen = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 8 - imgs.h DESDE ANIMACION
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-imgsh"
        ]
    ) as panel_imgs_animacion:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        imgs.h — ANIMACIÓN
                    </div>
                    """
                )

                gr.Markdown(
                    """
Selecciona preferiblemente un MP4 creado previamente por **Crear Animación** o **Adaptar video**.

El núcleo del conversor mantiene sin cambios la estructura crítica:

`const uint16_t frames[][60][100] PROGMEM = { ... };`
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    imgsh_anim_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Google Drive",
                        label="1. Origen de la animación"
                    )

                    imgsh_anim_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino de imgs.h"
                    )

                imgsh_anim_file_device = gr.File(
                    label=(
                        "Arrastra el MP4 aquí "
                        "o haz clic para buscarlo"
                    ),
                    file_count="single",
                    file_types=[
                        "video",
                        ".mp4"
                    ],
                    type="filepath",
                    visible=False
                )

                imgsh_anim_file_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="single",
                    label=(
                        "Seleccionar MP4 desde TESIS_POV"
                    ),
                    height=280,
                    visible=True
                )

                imgsh_anim_mode = gr.Radio(
                    choices=[
                        "Normal",
                        "Optimizado POV"
                    ],
                    value="Optimizado POV",
                    label="3. Modo de conversión POV"
                )

                gr.Markdown(
                    """
**Normal:** conserva el muestreo original.

**Optimizado POV:** recomendado para contenido sobre fondo negro. Reencuadra el área útil, prioriza el primer plano frente al fondo y aplica antialiasing 3×3 ponderado.
                    """,
                    elem_classes=["tool-info"]
                )

                imgsh_anim_generate = gr.Button(
                    "GENERAR imgs.h",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    imgsh_anim_status = gr.Markdown()

                    imgsh_anim_download = gr.File(
                        label="Descargar imgs.h",
                        interactive=False
                    )

                    imgsh_anim_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_imgs_animacion = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 9 - imgs.h DESDE IMAGEN
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-imgsh"
        ]
    ) as panel_imgs_imagen:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        imgs.h — IMAGEN
                    </div>
                    """
                )

                gr.Markdown(
                    """
Usa preferiblemente un **BMP 200×200** generado previamente por **Adaptar imagen**.

El núcleo del conversor mantiene sin cambios la estructura crítica:

`const uint16_t frame5[][100] PROGMEM = { ... };`
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    imgsh_image_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Google Drive",
                        label="1. Origen de la imagen"
                    )

                    imgsh_image_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino de imgs.h"
                    )

                imgsh_image_file_device = gr.File(
                    label=(
                        "Arrastra la imagen aquí "
                        "o haz clic para buscarla"
                    ),
                    file_count="single",
                    file_types=[
                        "image",
                        ".bmp",
                        ".png",
                        ".jpg",
                        ".jpeg",
                        ".webp",
                        ".tif",
                        ".tiff"
                    ],
                    type="filepath",
                    visible=False
                )

                imgsh_image_file_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="single",
                    label=(
                        "Seleccionar imagen desde TESIS_POV"
                    ),
                    height=280,
                    visible=True
                )

                imgsh_image_mode = gr.Radio(
                    choices=[
                        "Normal",
                        "Optimizado POV"
                    ],
                    value="Optimizado POV",
                    label="3. Modo de conversión POV"
                )

                gr.Markdown(
                    """
**Normal:** conserva el muestreo original.

**Optimizado POV:** recomendado para logos, texto o figuras sobre fondo negro. Reencuadra el contenido, le da prioridad al primer plano y aplica antialiasing 3×3 ponderado.
                    """,
                    elem_classes=["tool-info"]
                )

                imgsh_image_generate = gr.Button(
                    "GENERAR imgs.h",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    imgsh_image_status = gr.Markdown()

                    imgsh_image_download = gr.File(
                        label="Descargar imgs.h",
                        interactive=False
                    )

                    imgsh_image_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_imgs_imagen = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # LISTA DE PANELES
    # ========================================================

    PANELES = [
        panel_principal,
        panel_animaciones,
        panel_imagenes,
        panel_imgs,
        panel_manual,
        panel_stopmotion,
        panel_video,
        panel_adaptar_imagen,
        panel_imgs_animacion,
        panel_imgs_imagen
    ]


    # ========================================================
    # NAVEGACION PRINCIPAL
    # ========================================================

    btn_animaciones.click(
        ir_animaciones,
        outputs=PANELES
    )

    btn_imagenes.click(
        ir_imagenes,
        outputs=PANELES
    )

    btn_imgs.click(
        ir_imgs,
        outputs=PANELES
    )

    btn_manual.click(
        ir_manual,
        outputs=PANELES
    )


    # ========================================================
    # ANIMACIONES
    # ========================================================

    btn_stopmotion.click(
        ir_stopmotion,
        outputs=PANELES
    )

    btn_video.click(
        ir_video,
        outputs=PANELES
    )

    btn_back_animaciones.click(
        ir_principal,
        outputs=PANELES
    )

    stop_origen.change(
        mostrar_origen,
        inputs=[stop_origen],
        outputs=[
            stop_files_device,
            stop_files_drive
        ]
    )

    stop_generate.click(
        generar_stop_motion,
        inputs=[
            stop_origen,
            stop_files_device,
            stop_files_drive,
            stop_destino,
            stop_mode,
            stop_name
        ],
        outputs=[
            stop_status,
            stop_preview,
            stop_download,
            stop_drive_path
        ]
    )

    video_origen.change(
        mostrar_origen,
        inputs=[video_origen],
        outputs=[
            video_file_device,
            video_file_drive
        ]
    )

    video_generate.click(
        convertir_video_pov,
        inputs=[
            video_origen,
            video_file_device,
            video_file_drive,
            video_destino,
            video_mode,
            video_name
        ],
        outputs=[
            video_status,
            video_preview,
            video_download,
            video_drive_path
        ]
    )

    btn_back_stop.click(
        ir_animaciones,
        outputs=PANELES
    )

    btn_back_video.click(
        ir_animaciones,
        outputs=PANELES
    )


    # ========================================================
    # IMAGENES
    # ========================================================

    btn_adaptar_imagen.click(
        ir_adaptar_imagen,
        outputs=PANELES
    )

    btn_back_imagenes.click(
        ir_principal,
        outputs=PANELES
    )

    image_origen.change(
        mostrar_origen,
        inputs=[image_origen],
        outputs=[
            image_file_device,
            image_file_drive
        ]
    )

    image_generate.click(
        convertir_imagen_interfaz,
        inputs=[
            image_origen,
            image_file_device,
            image_file_drive,
            image_destino,
            image_mode,
            image_name
        ],
        outputs=[
            image_status,
            image_original_preview,
            image_result_preview,
            image_download,
            image_drive_path
        ]
    )

    btn_back_adaptar_imagen.click(
        ir_imagenes,
        outputs=PANELES
    )


    # ========================================================
    # imgs.h
    # ========================================================

    btn_imgs_animacion.click(
        ir_imgs_animacion,
        outputs=PANELES
    )

    btn_imgs_imagen.click(
        ir_imgs_imagen,
        outputs=PANELES
    )

    btn_back_imgs.click(
        ir_principal,
        outputs=PANELES
    )

    imgsh_anim_origen.change(
        mostrar_origen,
        inputs=[imgsh_anim_origen],
        outputs=[
            imgsh_anim_file_device,
            imgsh_anim_file_drive
        ]
    )

    imgsh_anim_generate.click(
        generar_imgsh_animacion,
        inputs=[
            imgsh_anim_origen,
            imgsh_anim_file_device,
            imgsh_anim_file_drive,
            imgsh_anim_destino,
            imgsh_anim_mode
        ],
        outputs=[
            imgsh_anim_status,
            imgsh_anim_download,
            imgsh_anim_drive_path
        ]
    )

    btn_back_imgs_animacion.click(
        ir_imgs,
        outputs=PANELES
    )

    imgsh_image_origen.change(
        mostrar_origen,
        inputs=[imgsh_image_origen],
        outputs=[
            imgsh_image_file_device,
            imgsh_image_file_drive
        ]
    )

    imgsh_image_generate.click(
        generar_imgsh_imagen,
        inputs=[
            imgsh_image_origen,
            imgsh_image_file_device,
            imgsh_image_file_drive,
            imgsh_image_destino,
            imgsh_image_mode
        ],
        outputs=[
            imgsh_image_status,
            imgsh_image_download,
            imgsh_image_drive_path
        ]
    )

    btn_back_imgs_imagen.click(
        ir_imgs,
        outputs=PANELES
    )


    # ========================================================
    # MANUAL
    # ========================================================

    btn_back_manual.click(
        ir_principal,
        outputs=PANELES
    )


# ============================================================
# INICIAR APP
# ============================================================
# Publicar la interfaz Gradio desde el entorno de Colab.
# inline=False evita incrustar la aplicación debajo de la celda y favorece uso
# en una pestaña independiente. El enlace compartido permanece ligado a la sesión.

resultado = app.launch(
    share=True,
    inline=False,
    prevent_thread_lock=True,
    show_error=True,
    max_file_size="500mb"
)


# ============================================================
# ENCONTRAR URL PUBLICA
# ============================================================
# Recuperar la dirección temporal generada por Gradio para abrir POV Creator.
# La dirección deja de funcionar al finalizar o reiniciar la sesión de Colab.

share_url = getattr(
    app,
    "share_url",
    None
)

if not share_url:
    try:
        if isinstance(resultado, tuple):
            for elemento in resultado:
                if (
                    isinstance(elemento, str)
                    and
                    "gradio.live" in elemento
                ):
                    share_url = elemento
                    break
    except Exception:
        pass


# ============================================================
# BOTON PARA ABRIR
# ============================================================
# Presentar un acceso directo desde el notebook hacia la interfaz independiente.
# Mantener el notebook como entorno de ejecución y la pestaña Gradio como interfaz.

if share_url:
    print()
    print("==============================================")
    print(" POV CREATOR ACTIVO")
    print("==============================================")
    print()
    print(share_url)

    display(
        HTML(
            f"""
            <div
                style="
                    text-align:center;
                    margin:25px;
                "
            >
                <a
                    href="{share_url}"
                    target="_blank"
                    style="
                        display:inline-block;
                        padding:16px 34px;
                        background:
                            linear-gradient(
                                90deg,
                                #073E82,
                                #0099E8
                            );
                        color:white;
                        text-decoration:none;
                        font-family:Arial,sans-serif;
                        font-weight:bold;
                        font-size:16px;
                        border-radius:8px;
                        box-shadow:
                            0 0 18px
                            rgba(0,150,255,0.65);
                    "
                >
                    🚀 ABRIR POV CREATOR
                </a>
            </div>
            """
        )
    )

else:
    print()
    print("✅ POV Creator está ejecutándose.")
    print(
        "Abre manualmente el enlace "
        "gradio.live mostrado arriba."
    )

Mounted at /content/drive

 VALIDANDO ASSETS

✅ Main
✅ Video
✅ Image
✅ imgsh
✅ Config


/tmp/ipykernel_2023/3214722693.py:3006: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4d19fa316845aa1905.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)

 POV CREATOR ACTIVO

https://4d19fa316845aa1905.gradio.live


Versiones de prueba

Version vieja

In [ ]:
# ============================================================
# POV CREATOR - V0.9
# Google Colab + Gradio
# ============================================================
#
# DOCUMENTACIÓN TÉCNICA PARA TESIS
#
# Finalidad general
# -----------------
# Preparar contenido gráfico para una pantalla POV basada en ESP32.
# Concentrar en una sola interfaz las etapas de normalización de
# imágenes, creación de animaciones, adaptación de videos y generación
# de los archivos imgs.h utilizados por el firmware embebido.
#
# Relación software - hardware
# ----------------------------
# Convertir contenido visual convencional a una representación adecuada
# para visualización por persistencia de la visión. Trabajar con una
# superficie lógica de 200 x 200 píxeles y transformar posteriormente
# esa información cartesiana a muestras polares compatibles con el
# barrido angular del sistema físico.
#
# Resolución angular utilizada: 6 grados.
# Vueltas lógicas por imagen: 360 / 6 = 60 posiciones angulares.
# Información radial por posición: 100 valores.
# Formato de color final: RGB565 de 16 bits.
#
# La matriz 60 x 100 constituye la representación intermedia utilizada
# por el firmware para reconstruir la imagen durante la rotación.
#
# Flujo principal
# ---------------
# Imagen / video / secuencia de imágenes
#                  |
#                  v
#        Normalización 200 x 200
#                  |
#                  v
#       Conversión temporal / espacial
#                  |
#                  v
#             Generación imgs.h
#                  |
#                  v
#              Firmware ESP32
#                  |
#                  v
#             Visualización POV
#
# Restricción de memoria
# ----------------------
# Mantener un máximo de 69 frames almacenados para animación. Cada frame
# utiliza aproximadamente 12 000 bytes: 60 filas x 100 valores x 2 bytes.
# El límite permite trabajar dentro de la partición de aplicación cercana
# a 2 MB configurada para conservar actualización OTA.
#
# Criterio de integridad
# ----------------------
# Mantener sin alteraciones funcionales la geometría polar, el empaquetado
# RGB565 y las estructuras frame5[][100] y frames[][60][100]. Estos bloques
# representan la interfaz directa entre el software de preparación y el
# firmware del sistema POV.
#
# Módulos funcionales
# -------------------
#   - Crear Animación
#   - Adaptar video
#   - Adaptar imagen
#   - Generar imgs.h desde animación
#   - Generar imgs.h desde imagen
#
# Nota de mantenimiento
# ---------------------
# Conservar nombres internos como STOPMOTION_DIR, variables stop_* y rutas
# históricas para evitar cambios innecesarios en el comportamiento. El
# nombre visible de la herramienta pasa a ser CREAR ANIMACIÓN.
# ============================================================

import sys
import subprocess

# Instalar/actualizar Gradio desde Python para que este archivo
# también pueda ejecutarse con %run en Google Colab.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-U", "gradio"],
    check=True
)

import gradio as gr
import os
import re
import cv2
import json
import math
import base64
import shutil
import tempfile
import mimetypes

from PIL import Image, ImageOps
from google.colab import drive
from IPython.display import display, HTML


# ============================================================
# MONTAR GOOGLE DRIVE
# ============================================================
# Persistencia de archivos fuera de la sesión temporal de Colab.
# Mantener recursos gráficos, videos procesados, imágenes normalizadas,
# fondos de interfaz y archivos imgs.h dentro de una estructura común.
# Evitar dependencia del almacenamiento efímero del entorno de ejecución.

drive.mount("/content/drive")


# ============================================================
# CONFIGURACION POV
# ============================================================
# Parámetros comunes para toda la cadena de procesamiento.
# 200 x 200: formato cartesiano normalizado antes de conversión polar.
# 10 FPS: frecuencia de preparación de animaciones y videos.
# 69 frames: límite práctico de almacenamiento para animación con OTA.
# Fondo negro: ausencia de emisión luminosa en zonas transparentes o libres.

ANCHO = 200
ALTO = 200
FPS = 10
TOTAL_FRAMES_FINAL = 69
COLOR_FONDO = (0, 0, 0)


# ============================================================
# RUTAS
# ============================================================
# Separación entre almacenamiento permanente y procesamiento temporal.
# Drive: resultados reutilizables entre sesiones.
# TEMP_ROOT: archivos intermedios descartables durante cada conversión.
# La organización por carpetas facilita trazabilidad y repetición de pruebas.

BASE_DIR = "/content/drive/MyDrive/TESIS_POV"
PROGRAMA_DIR = os.path.join(BASE_DIR, "Programa")
ASSETS_DIR = os.path.join(PROGRAMA_DIR, "assets")
VIDEOS_DIR = os.path.join(BASE_DIR, "Videos")
STOPMOTION_DIR = os.path.join(BASE_DIR, "StopMotion")
IMAGES_DIR = os.path.join(BASE_DIR, "Imagenes")
SALIDA_DIR = os.path.join(BASE_DIR, "Salida")

TEMP_ROOT = os.path.join(
    tempfile.gettempdir(),
    "POV_CREATOR"
)

for carpeta in [
    PROGRAMA_DIR,
    ASSETS_DIR,
    VIDEOS_DIR,
    STOPMOTION_DIR,
    IMAGES_DIR,
    SALIDA_DIR,
    TEMP_ROOT
]:
    os.makedirs(carpeta, exist_ok=True)


# ============================================================
# FONDOS
# ============================================================
# Recursos visuales independientes para distinguir cada módulo de la interfaz.
# El cambio de fondo funciona únicamente como capa de presentación y no altera
# parámetros de conversión, geometría POV ni archivos generados.

MAIN_BG = os.path.join(ASSETS_DIR, "Main.png")
VIDEO_BG = os.path.join(ASSETS_DIR, "Video.png")
IMAGE_BG = os.path.join(ASSETS_DIR, "Image.png")
IMGSH_BG = os.path.join(ASSETS_DIR, "imgsh.png")
CONFIG_BG = os.path.join(ASSETS_DIR, "Config.png")

FONDOS = {
    "Main": MAIN_BG,
    "Video": VIDEO_BG,
    "Image": IMAGE_BG,
    "imgsh": IMGSH_BG,
    "Config": CONFIG_BG
}

print()
print("==============================================")
print(" VALIDANDO ASSETS")
print("==============================================")
print()

for nombre, ruta in FONDOS.items():
    if not os.path.isfile(ruta):
        raise Exception(
            f"No se encontró el fondo '{nombre}'.\n\n"
            f"Ruta esperada:\n{ruta}"
        )
    print(f"✅ {nombre}")


# ============================================================
# COMPROBAR FFMPEG
# ============================================================
# Validación previa de la herramienta encargada de codificación de video.
# FFmpeg permite normalizar resolución, FPS, códec, formato de píxel y audio.
# H.264 + YUV420P favorece compatibilidad con navegadores, Drive y mensajería.

try:
    subprocess.run(
        ["ffmpeg", "-version"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True
    )
except Exception:
    raise Exception("FFmpeg no está disponible en Colab.")


# ============================================================
# FONDOS BASE64
# ============================================================
# Incrustar los fondos directamente dentro del CSS de la aplicación.
# Evitar dependencias de rutas locales al abrir Gradio mediante enlace público.
# Mantener intactos los archivos originales almacenados en Google Drive.

def imagen_a_base64(ruta):
    mime, _ = mimetypes.guess_type(ruta)
    if mime is None:
        mime = "image/png"

    with open(ruta, "rb") as archivo:
        contenido = base64.b64encode(
            archivo.read()
        ).decode("utf-8")

    return f"data:{mime};base64,{contenido}"


MAIN_DATA = imagen_a_base64(MAIN_BG)
VIDEO_DATA = imagen_a_base64(VIDEO_BG)
IMAGE_DATA = imagen_a_base64(IMAGE_BG)
IMGSH_DATA = imagen_a_base64(IMGSH_BG)
CONFIG_DATA = imagen_a_base64(CONFIG_BG)


# ============================================================
# UTILIDADES
# ============================================================
# Funciones auxiliares para nombres, rutas, selección de archivos y destinos.
# Centralizar validaciones antes de entrar a los algoritmos de conversión.
# Reducir errores por caracteres inválidos, rutas inexistentes o archivos fuera
# de la carpeta de trabajo definida para TESIS_POV.

def limpiar_nombre(nombre):
    if nombre is None:
        nombre = ""

    nombre = nombre.strip()

    if not nombre:
        raise ValueError("Debes escribir un nombre para el archivo.")

    nombre = re.sub(r'[\\/:*?"<>|]', "_", nombre)
    nombre = nombre.strip(" .")

    if not nombre:
        raise ValueError("El nombre del archivo no es válido.")

    return nombre[:100]


def orden_natural(texto):
    texto = os.path.basename(texto)

    return [
        int(parte) if parte.isdigit() else parte.lower()
        for parte in re.split(r"(\d+)", texto)
    ]


def obtener_lista_rutas(valor):
    if valor is None:
        return []

    if not isinstance(valor, list):
        valor = [valor]

    rutas = []

    for elemento in valor:
        if elemento is None:
            continue

        if isinstance(elemento, str):
            ruta = elemento
        elif hasattr(elemento, "path"):
            ruta = elemento.path
        elif hasattr(elemento, "name"):
            ruta = elemento.name
        else:
            continue

        rutas.append(str(ruta))

    return rutas


def normalizar_ruta_drive(ruta):
    if not ruta:
        raise ValueError("Ruta de Drive vacía.")

    if os.path.isabs(ruta):
        ruta_absoluta = os.path.realpath(ruta)
    else:
        ruta_absoluta = os.path.realpath(
            os.path.join(BASE_DIR, ruta)
        )

    base_real = os.path.realpath(BASE_DIR)

    try:
        pertenece = (
            os.path.commonpath(
                [ruta_absoluta, base_real]
            )
            == base_real
        )
    except Exception:
        pertenece = False

    if not pertenece:
        raise ValueError(
            "El archivo seleccionado está fuera de TESIS_POV."
        )

    if not os.path.isfile(ruta_absoluta):
        raise ValueError(
            f"No se encontró el archivo:\n{ruta_absoluta}"
        )

    return ruta_absoluta


def resolver_entrada(
    origen,
    archivos_dispositivo,
    archivos_drive
):
    if origen == "Dispositivo":
        rutas = obtener_lista_rutas(
            archivos_dispositivo
        )

    elif origen == "Google Drive":
        rutas_originales = obtener_lista_rutas(
            archivos_drive
        )

        rutas = [
            normalizar_ruta_drive(ruta)
            for ruta in rutas_originales
        ]

    else:
        raise ValueError(
            "Origen de archivos no válido."
        )

    rutas = [
        ruta
        for ruta in rutas
        if ruta and os.path.isfile(ruta)
    ]

    if not rutas:
        raise ValueError(
            "No seleccionaste ningún archivo."
        )

    return rutas


def crear_trabajo_temporal(prefijo):
    return tempfile.mkdtemp(
        prefix=prefijo + "_",
        dir=TEMP_ROOT
    )


def mostrar_origen(origen):
    dispositivo = origen == "Dispositivo"

    return (
        gr.update(visible=dispositivo),
        gr.update(visible=not dispositivo)
    )


def destino_flags(destino):
    guardar_drive = destino in [
        "Google Drive",
        "Drive + descarga"
    ]

    permitir_descarga = destino in [
        "Descargar al dispositivo",
        "Drive + descarga"
    ]

    return guardar_drive, permitir_descarga


# ============================================================
# CONVERSION DE IMAGEN 200x200
# Lógica compartida por Crear Animación y Adaptar Imagen.
# ============================================================
# Normalización espacial antes de cualquier conversión POV.
# RECORTAR: conservar proporción y llenar 200 x 200 eliminando excedentes.
# AJUSTAR: conservar todo el contenido y completar espacios con negro.
# ESTIRAR: ocupar 200 x 200 sin conservar necesariamente la proporción.
# La conversión final a RGB elimina ambigüedades de alfa o modos de color.

def convertir_imagen_pov(ruta, modo):
    imagen = Image.open(ruta)
    formato_original = imagen.format
    tamano_original = imagen.size
    modo_original = imagen.mode

    imagen = ImageOps.exif_transpose(imagen)

    if imagen.mode in ("RGBA", "LA", "P"):
        imagen_rgba = imagen.convert("RGBA")

        fondo = Image.new(
            "RGB",
            imagen_rgba.size,
            COLOR_FONDO
        )

        fondo.paste(
            imagen_rgba,
            mask=imagen_rgba.getchannel("A")
        )

        imagen = fondo
    else:
        imagen = imagen.convert("RGB")

    if modo == "RECORTAR":
        imagen_final = ImageOps.fit(
            imagen,
            (ANCHO, ALTO),
            method=Image.Resampling.LANCZOS,
            centering=(0.5, 0.5)
        )

    elif modo == "AJUSTAR":
        imagen_redimensionada = ImageOps.contain(
            imagen,
            (ANCHO, ALTO),
            method=Image.Resampling.LANCZOS
        )

        imagen_final = Image.new(
            "RGB",
            (ANCHO, ALTO),
            COLOR_FONDO
        )

        x = (
            ANCHO -
            imagen_redimensionada.width
        ) // 2

        y = (
            ALTO -
            imagen_redimensionada.height
        ) // 2

        imagen_final.paste(
            imagen_redimensionada,
            (x, y)
        )

    elif modo == "ESTIRAR":
        imagen_final = imagen.resize(
            (ANCHO, ALTO),
            Image.Resampling.LANCZOS
        )

    else:
        raise ValueError(
            "Modo inválido. Usa RECORTAR, AJUSTAR o ESTIRAR."
        )

    imagen_final = imagen_final.convert("RGB")

    return (
        imagen_final,
        formato_original,
        tamano_original,
        modo_original
    )


# ============================================================
# FFPROBE
# ============================================================
# Lectura técnica del video antes y después de la conversión.
# Verificar resolución, duración, FPS, códec, formato de píxel y número de frames.
# Utilizar estos datos como control de calidad sin modificar el contenido visual.

def analizar_video(ruta, contar_frames=True):
    comando = [
        "ffprobe",
        "-v", "error",
        "-select_streams", "v:0"
    ]

    if contar_frames:
        comando.append("-count_frames")

    comando.extend(
        [
            "-show_entries",
            (
                "stream="
                "codec_name,"
                "width,"
                "height,"
                "r_frame_rate,"
                "pix_fmt,"
                "nb_read_frames:"
                "format=duration"
            ),
            "-of", "json",
            ruta
        ]
    )

    resultado = subprocess.run(
        comando,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if resultado.returncode != 0:
        raise ValueError(
            "FFprobe no pudo analizar el video."
        )

    datos = json.loads(resultado.stdout)
    streams = datos.get("streams", [])

    if not streams:
        raise ValueError(
            "El archivo no contiene una pista de video válida."
        )

    stream = streams[0]

    duracion = datos.get(
        "format",
        {}
    ).get(
        "duration",
        "0"
    )

    try:
        duracion = float(duracion)
    except Exception:
        duracion = 0.0

    return {
        "codec": stream.get(
            "codec_name",
            "desconocido"
        ),
        "width": int(
            stream.get(
                "width",
                0
            ) or 0
        ),
        "height": int(
            stream.get(
                "height",
                0
            ) or 0
        ),
        "fps": stream.get(
            "r_frame_rate",
            "desconocido"
        ),
        "pix_fmt": stream.get(
            "pix_fmt",
            "desconocido"
        ),
        "frames": stream.get(
            "nb_read_frames",
            "desconocido"
        ),
        "duration": duracion
    }


# ============================================================
# CREAR ANIMACIÓN
# ============================================================
# Construcción de una animación a partir de 1 a 69 imágenes estáticas.
# Distribución temporal automática hasta completar exactamente 69 frames.
# 69 imágenes: una aparición por imagen.
# Menos de 69: repetición equilibrada para ocupar toda la secuencia.
# Salida normalizada: MP4, 200 x 200, 10 FPS, H.264, YUV420P y sin audio.
# La carpeta interna StopMotion permanece por compatibilidad con la estructura
# de almacenamiento ya utilizada durante el desarrollo.

def guardar_stopmotion_en_drive(
    nombre,
    carpeta_frames_temporal,
    video_temporal_final
):
    proyecto_drive = os.path.join(
        STOPMOTION_DIR,
        nombre
    )

    frames_drive = os.path.join(
        proyecto_drive,
        "Imagenes_200x200"
    )

    video_drive = os.path.join(
        VIDEOS_DIR,
        nombre + ".mp4"
    )

    if os.path.exists(proyecto_drive):
        shutil.rmtree(proyecto_drive)

    os.makedirs(
        proyecto_drive,
        exist_ok=True
    )

    shutil.copytree(
        carpeta_frames_temporal,
        frames_drive
    )

    shutil.copy2(
        video_temporal_final,
        video_drive
    )

    return (
        proyecto_drive,
        frames_drive,
        video_drive
    )


def generar_stop_motion(
    origen,
    archivos_dispositivo,
    archivos_drive,
    destino,
    modo,
    nombre_video,
    progress=gr.Progress()
):
    try:
        nombre_video = limpiar_nombre(
            nombre_video
        )

        rutas = resolver_entrada(
            origen,
            archivos_dispositivo,
            archivos_drive
        )

        rutas = sorted(
            rutas,
            key=orden_natural
        )

        cantidad_imagenes = len(rutas)

        if cantidad_imagenes < 1:
            raise ValueError(
                "Debes seleccionar al menos una imagen."
            )

        if cantidad_imagenes > TOTAL_FRAMES_FINAL:
            raise ValueError(
                f"Seleccionaste {cantidad_imagenes} imágenes.\n\n"
                f"El máximo permitido es {TOTAL_FRAMES_FINAL}."
            )

        trabajo = crear_trabajo_temporal(
            "stopmotion"
        )

        carpeta_frames = os.path.join(
            trabajo,
            "Imagenes_200x200"
        )

        os.makedirs(
            carpeta_frames,
            exist_ok=True
        )

        imagenes_convertidas = []

        for indice, ruta in enumerate(
            rutas,
            start=1
        ):
            progress(
                (indice / cantidad_imagenes) * 0.35,
                desc=(
                    f"Convirtiendo imagen "
                    f"{indice}/{cantidad_imagenes}"
                )
            )

            try:
                imagen_final, _, _, _ = (
                    convertir_imagen_pov(
                        ruta,
                        modo
                    )
                )
            except Exception as error:
                raise ValueError(
                    "No se pudo procesar:\n"
                    f"{os.path.basename(ruta)}\n\n"
                    f"{error}"
                )

            nombre_bmp = (
                f"imagen_{indice:03d}.bmp"
            )

            ruta_bmp = os.path.join(
                carpeta_frames,
                nombre_bmp
            )

            imagen_final.save(
                ruta_bmp,
                "BMP"
            )

            imagenes_convertidas.append(
                ruta_bmp
            )

        if cantidad_imagenes == TOTAL_FRAMES_FINAL:
            repeticiones = [
                1
                for _ in range(
                    cantidad_imagenes
                )
            ]
        else:
            repeticiones_base = (
                TOTAL_FRAMES_FINAL
                // cantidad_imagenes
            )

            sobrantes = (
                TOTAL_FRAMES_FINAL
                % cantidad_imagenes
            )

            repeticiones = []

            for i in range(
                cantidad_imagenes
            ):
                cantidad = repeticiones_base

                if i < sobrantes:
                    cantidad += 1

                repeticiones.append(
                    cantidad
                )

        if sum(repeticiones) != TOTAL_FRAMES_FINAL:
            raise ValueError(
                "Error calculando los 69 frames."
            )

        video_mp4v = os.path.join(
            trabajo,
            nombre_video + "_TEMP.mp4"
        )

        fourcc = cv2.VideoWriter_fourcc(
            *"mp4v"
        )

        escritor = cv2.VideoWriter(
            video_mp4v,
            fourcc,
            FPS,
            (ANCHO, ALTO)
        )

        if not escritor.isOpened():
            raise ValueError(
                "No se pudo iniciar VideoWriter."
            )

        contador = 0

        for i, ruta_bmp in enumerate(
            imagenes_convertidas
        ):
            imagen_cv = cv2.imread(
                ruta_bmp
            )

            if imagen_cv is None:
                escritor.release()

                raise ValueError(
                    "No se pudo leer:\n"
                    f"{ruta_bmp}"
                )

            if (
                imagen_cv.shape[1] != ANCHO
                or
                imagen_cv.shape[0] != ALTO
            ):
                imagen_cv = cv2.resize(
                    imagen_cv,
                    (ANCHO, ALTO),
                    interpolation=cv2.INTER_LANCZOS4
                )

            for _ in range(
                repeticiones[i]
            ):
                escritor.write(
                    imagen_cv
                )

                contador += 1

                progress(
                    0.35
                    +
                    (
                        contador
                        /
                        TOTAL_FRAMES_FINAL
                    )
                    * 0.35,
                    desc=(
                        f"Creando frame "
                        f"{contador}/"
                        f"{TOTAL_FRAMES_FINAL}"
                    )
                )

        escritor.release()

        if not os.path.isfile(video_mp4v):
            raise ValueError(
                "No se creó el video temporal."
            )

        video_final = os.path.join(
            trabajo,
            nombre_video + ".mp4"
        )

        progress(
            0.75,
            desc="Codificando H.264"
        )

        comando = [
            "ffmpeg",
            "-y",
            "-i", video_mp4v,
            "-an",
            "-c:v", "libx264",
            "-profile:v", "baseline",
            "-level", "3.0",
            "-pix_fmt", "yuv420p",
            "-r", str(FPS),
            "-crf", "18",
            "-preset", "medium",
            "-movflags", "+faststart",
            video_final
        ]

        resultado = subprocess.run(
            comando,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if resultado.returncode != 0:
            raise ValueError(
                "FFmpeg no pudo crear el video H.264.\n\n"
                +
                resultado.stderr[-1200:]
            )

        try:
            os.remove(video_mp4v)
        except Exception:
            pass

        progress(
            0.90,
            desc="Verificando video"
        )

        info = analizar_video(
            video_final
        )

        try:
            frames_finales = int(
                info["frames"]
            )
        except Exception:
            frames_finales = 0

        if frames_finales != 69:
            raise ValueError(
                "El video final no contiene "
                "exactamente 69 frames.\n\n"
                f"Detectados: {frames_finales}"
            )

        if (
            info["width"] != 200
            or
            info["height"] != 200
        ):
            raise ValueError(
                "La resolución final no es 200 x 200."
            )

        if info["codec"] != "h264":
            raise ValueError(
                "El codec final no es H.264."
            )

        if info["pix_fmt"] != "yuv420p":
            raise ValueError(
                "El formato final no es YUV420P."
            )

        guardar_drive, permitir_descarga = (
            destino_flags(destino)
        )

        ruta_drive = ""
        frames_drive = ""

        if guardar_drive:
            progress(
                0.95,
                desc="Guardando en Google Drive"
            )

            (
                _,
                frames_drive,
                ruta_drive
            ) = guardar_stopmotion_en_drive(
                nombre_video,
                carpeta_frames,
                video_final
            )

        tamano_mb = (
            os.path.getsize(video_final)
            /
            (1024 * 1024)
        )

        distribucion = ", ".join(
            str(x)
            for x in repeticiones
        )

        if len(distribucion) > 180:
            distribucion = (
                distribucion[:180]
                + "..."
            )

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Google Drive:**  \n"
                f"`{ruta_drive}`"
                "\n\n"
                "🖼️ **Frames 200×200:**  \n"
                f"`{frames_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar MP4** "
                "para guardar el archivo en tu dispositivo."
            )

        estado = f"""
### ✅ ANIMACIÓN GENERADA

**Imágenes originales:** {cantidad_imagenes}
**Frames finales:** {frames_finales}
**Resolución:** {info["width"]} × {info["height"]}
**FPS:** 10
**Duración:** {info["duration"]:.2f} s
**Codec:** {info["codec"]}
**Pixel format:** {info["pix_fmt"]}
**Tamaño:** {tamano_mb:.2f} MB

**Distribución de frames:**
`{distribucion}`

{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            video_final
            if permitir_descarga
            else None
        )

        return (
            estado,
            video_final,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))


# ============================================================
# ADAPTAR VIDEO
# ============================================================
# Normalización de videos existentes al formato de entrada del sistema POV.
# Conservar aproximadamente la duración original mediante conversión a 10 FPS.
# No limitar aquí a 69 frames: aplicar la selección uniforme más adelante, dentro
# del generador imgs.h de animación, conservando la relación temporal original.

def guardar_video_en_drive(
    nombre,
    video_temporal_final
):
    video_drive = os.path.join(
        VIDEOS_DIR,
        nombre + ".mp4"
    )

    shutil.copy2(
        video_temporal_final,
        video_drive
    )

    return video_drive


def convertir_video_pov(
    origen,
    archivo_dispositivo,
    archivo_drive,
    destino,
    modo,
    nombre_video,
    progress=gr.Progress()
):
    try:
        nombre_video = limpiar_nombre(
            nombre_video
        )

        rutas = resolver_entrada(
            origen,
            archivo_dispositivo,
            archivo_drive
        )

        if len(rutas) != 1:
            raise ValueError(
                "Selecciona solamente un video."
            )

        input_video = rutas[0]

        progress(
            0.10,
            desc="Analizando video original"
        )

        original = analizar_video(
            input_video,
            contar_frames=False
        )

        if modo == "RECORTAR":
            filtro = (
                f"scale="
                f"{ANCHO}:{ALTO}:"
                f"force_original_aspect_ratio=increase,"
                f"crop="
                f"{ANCHO}:{ALTO},"
                f"fps={FPS},"
                f"setsar=1"
            )

        elif modo == "AJUSTAR":
            filtro = (
                f"scale="
                f"{ANCHO}:{ALTO}:"
                f"force_original_aspect_ratio=decrease,"
                f"pad="
                f"{ANCHO}:{ALTO}:"
                f"(ow-iw)/2:"
                f"(oh-ih)/2:"
                f"black,"
                f"fps={FPS},"
                f"setsar=1"
            )

        elif modo == "ESTIRAR":
            filtro = (
                f"scale="
                f"{ANCHO}:{ALTO},"
                f"fps={FPS},"
                f"setsar=1"
            )

        else:
            raise ValueError(
                "Modo de ajuste desconocido."
            )

        trabajo = crear_trabajo_temporal(
            "video"
        )

        output_video = os.path.join(
            trabajo,
            nombre_video + ".mp4"
        )

        progress(
            0.25,
            desc="Convirtiendo video"
        )

        comando = [
            "ffmpeg",
            "-y",
            "-i", input_video,
            "-vf", filtro,
            "-c:v", "libx264",
            "-profile:v", "baseline",
            "-level", "3.0",
            "-pix_fmt", "yuv420p",
            "-crf", "18",
            "-preset", "medium",
            "-an",
            "-movflags", "+faststart",
            output_video
        ]

        resultado = subprocess.run(
            comando,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True
        )

        if resultado.returncode != 0:
            raise ValueError(
                "FFmpeg no pudo convertir el video.\n\n"
                +
                resultado.stderr[-1200:]
            )

        progress(
            0.78,
            desc="Verificando formato"
        )

        final = analizar_video(
            output_video
        )

        if (
            final["width"] != 200
            or
            final["height"] != 200
        ):
            raise ValueError(
                "La resolución final no es 200 x 200."
            )

        if final["codec"] != "h264":
            raise ValueError(
                "El codec final no es H.264."
            )

        if final["pix_fmt"] != "yuv420p":
            raise ValueError(
                "El pixel format final no es YUV420P."
            )

        guardar_drive, permitir_descarga = (
            destino_flags(destino)
        )

        ruta_drive = ""

        if guardar_drive:
            progress(
                0.92,
                desc="Guardando en Google Drive"
            )

            ruta_drive = guardar_video_en_drive(
                nombre_video,
                output_video
            )

        tamano_mb = (
            os.path.getsize(output_video)
            /
            (1024 * 1024)
        )

        try:
            frames_finales = int(
                final["frames"]
            )
        except Exception:
            frames_finales = "desconocido"

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Guardado en Google Drive:**  \n"
                f"`{ruta_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar MP4** "
                "para guardarlo en tu dispositivo."
            )

        estado = f"""
### ✅ VIDEO CONVERTIDO CORRECTAMENTE

#### Original

**Archivo:** `{os.path.basename(input_video)}`
**Resolución:** {original["width"]} × {original["height"]}
**Codec:** {original["codec"]}
**FPS:** {original["fps"]}
**Duración:** {original["duration"]:.2f} s

#### Resultado POV

**Resolución:** {final["width"]} × {final["height"]}
**FPS:** {final["fps"]}
**Frames:** {frames_finales}
**Duración:** {final["duration"]:.2f} s
**Codec:** {final["codec"]}
**Pixel format:** {final["pix_fmt"]}
**Audio:** NO
**Fast Start:** SÍ
**Tamaño:** {tamano_mb:.2f} MB

{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            output_video
            if permitir_descarga
            else None
        )

        return (
            estado,
            output_video,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))


# ============================================================
# ADAPTAR IMAGEN
# ============================================================
# Preparación de una imagen individual para el conversor POV de imagen fija.
# Generar BMP de 200 x 200 píxeles en modo RGB y mantener fondo negro cuando
# exista transparencia o espacio libre. Este BMP actúa como entrada controlada
# para la generación posterior de frame5[][100].

def convertir_imagen_interfaz(
    origen,
    archivo_dispositivo,
    archivo_drive,
    destino,
    modo,
    nombre_salida,
    progress=gr.Progress()
):
    try:
        nombre_salida = limpiar_nombre(
            nombre_salida
        )

        rutas = resolver_entrada(
            origen,
            archivo_dispositivo,
            archivo_drive
        )

        if len(rutas) != 1:
            raise ValueError(
                "Selecciona solamente una imagen."
            )

        input_image = rutas[0]

        progress(
            0.20,
            desc="Abriendo imagen"
        )

        (
            imagen_final,
            formato_original,
            tamano_original,
            modo_original
        ) = convertir_imagen_pov(
            input_image,
            modo
        )

        trabajo = crear_trabajo_temporal(
            "imagen"
        )

        output_bmp = os.path.join(
            trabajo,
            nombre_salida + ".bmp"
        )

        progress(
            0.60,
            desc="Creando BMP 200×200"
        )

        imagen_final.save(
            output_bmp,
            "BMP"
        )

        # Validación final
        verificacion = Image.open(
            output_bmp
        )

        if verificacion.size != (200, 200):
            raise ValueError(
                "La imagen final no quedó en 200×200."
            )

        if verificacion.mode != "RGB":
            raise ValueError(
                "La imagen final no quedó en modo RGB."
            )

        guardar_drive, permitir_descarga = (
            destino_flags(destino)
        )

        ruta_drive = ""

        if guardar_drive:
            progress(
                0.82,
                desc="Guardando en Google Drive"
            )

            ruta_drive = os.path.join(
                IMAGES_DIR,
                nombre_salida + ".bmp"
            )

            shutil.copy2(
                output_bmp,
                ruta_drive
            )

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Guardado en Google Drive:**  \n"
                f"`{ruta_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar BMP** "
                "para guardarlo en tu dispositivo."
            )

        estado = f"""
### ✅ IMAGEN CONVERTIDA CORRECTAMENTE

#### Original

**Archivo:** `{os.path.basename(input_image)}`
**Formato:** {formato_original or "desconocido"}
**Resolución:** {tamano_original[0]} × {tamano_original[1]}
**Modo:** {modo_original}

#### Resultado POV

**Resolución:** 200 × 200
**Modo:** RGB
**Formato:** BMP
**Ajuste utilizado:** {modo}

{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            output_bmp
            if permitir_descarga
            else None
        )

        return (
            estado,
            input_image,
            output_bmp,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))


# ============================================================
# imgs.h DESDE ANIMACION
# ============================================================
# Puente entre video preparado y firmware ESP32.
# Seleccionar un máximo de 69 frames distribuidos uniformemente desde el primero
# hasta el último, evitando concentrar la reducción en una sola zona del video.
# Convertir cada muestra RGB888 a RGB565 para reducir memoria y coincidir con el
# formato esperado por la lógica de visualización embebida.
#
# Un frame almacenado ocupa aproximadamente:
# 60 posiciones angulares x 100 posiciones radiales x 2 bytes = 12 000 bytes.
#
# NUCLEO CRITICO:
# Se conserva la estructura del conversor suministrado:
#
# #define POV_MODE_ANIMATION
# #define POV_ORIGINAL_FRAMES ...
# #define POV_STORED_FRAMES ...
# const uint16_t frames[][60][100] PROGMEM = {
#
# También se conserva la geometría y RGB565.
# ============================================================

def crear_imgsh_animacion_sagrado(
    VIDEO_PATH,
    OUTPUT_PATH,
    progress=None
):
    # ==========================================
    # CONFIGURACION
    # ==========================================

    # Límite de almacenamiento compatible con la partición OTA empleada.
    # Evitar aumentar este valor sin revisar nuevamente el tamaño compilado.
    MAX_FRAMES = 69

    # Paso angular del sistema POV: 360 / 6 = 60 muestras por revolución.
    resolution = 6

    NUM_LEDS = 200
    HALF_LEDS = int(NUM_LEDS / 2)

    ROWS = int(360 / resolution)


    # ==========================================
    # ABRIR VIDEO
    # ==========================================

    cap = cv2.VideoCapture(VIDEO_PATH)

    if not cap.isOpened():
        raise Exception("No se pudo abrir el video")


    # ==========================================
    # INFORMACION DEL VIDEO
    # ==========================================

    TOTAL_FRAMES = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT)
    )

    FPS_VIDEO = cap.get(
        cv2.CAP_PROP_FPS
    )

    WIDTH = int(
        cap.get(cv2.CAP_PROP_FRAME_WIDTH)
    )

    HEIGHT = int(
        cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
    )


    # ==========================================
    # CALCULAR FRAMES A CONSERVAR
    # ==========================================

    if TOTAL_FRAMES <= MAX_FRAMES:

        STORED_FRAMES = TOTAL_FRAMES

        selected_frames = list(
            range(TOTAL_FRAMES)
        )

    else:

        STORED_FRAMES = MAX_FRAMES

        selected_frames = []

        for i in range(STORED_FRAMES):

            index = round(
                i *
                (TOTAL_FRAMES - 1) /
                (STORED_FRAMES - 1)
            )

            selected_frames.append(index)


    # ==========================================
    # CREAR ARCHIVO imgs.h
    # ==========================================

    f = open(
        OUTPUT_PATH,
        'w'
    )


    # ==========================================
    # CABECERA
    # ==========================================

    f.write(
        "//=====================================\n"
    )

    f.write(
        "// Archivo generado automaticamente\n"
    )

    f.write(
        "// Conversor POV Animacion\n"
    )

    f.write(
        "//=====================================\n\n"
    )


    f.write(
        "#pragma once\n"
    )

    f.write(
        "#include <Arduino.h>\n"
    )

    f.write(
        "#include <pgmspace.h>\n\n"
    )


    # ==========================================
    # IDENTIFICAR MODO ANIMACION
    # ==========================================

    f.write(
        "#define POV_MODE_ANIMATION\n\n"
    )


    # ==========================================
    # INFORMACION DE LA ANIMACION
    # ==========================================

    f.write(
        f"#define POV_ORIGINAL_FRAMES {TOTAL_FRAMES}\n"
    )

    f.write(
        f"#define POV_STORED_FRAMES {STORED_FRAMES}\n\n"
    )


    # ==========================================
    # ARRAY GENERAL
    # ==========================================

    f.write(
        "const uint16_t frames[][60][100] PROGMEM = {\n"
    )


    # ==========================================
    # CONVERTIR FRAMES
    # ==========================================

    stored_cont = 0

    video_frame_index = 0

    selected_position = 0


    while cap.isOpened():

        ret, frame = cap.read()


        if not ret:

            break


        # ======================================
        # ¿ESTE FRAME SE GUARDA?
        # ======================================

        if (
            selected_position < len(selected_frames)
            and
            video_frame_index ==
            selected_frames[selected_position]
        ):


            if progress is not None:
                avance = (
                    selected_position
                    /
                    max(
                        1,
                        len(selected_frames)
                    )
                )

                progress(
                    0.10 + avance * 0.80,
                    desc=(
                        "Procesando frame "
                        f"{selected_position + 1}/"
                        f"{len(selected_frames)}"
                    )
                )


            degrees = 0

            row = 0


            img_converted = [

                [
                    0
                    for k in range(HALF_LEDS)
                ]

                for l in range(ROWS)

            ]


            # ==================================
            # CONVERSION POLAR POV
            # ==================================

            while degrees < 360:


                for j in range(
                    0,
                    HALF_LEDS
                ):


                    # ==========================
                    # CUADRANTE 1
                    # ==========================

                    if degrees <= 90:

                        degrees_aux = (
                            90 -
                            degrees
                        )


                        x = int(

                            HALF_LEDS +

                            (
                                HALF_LEDS -
                                j
                            )

                            *

                            math.sin(
                                math.radians(
                                    degrees_aux
                                )
                            )

                        )


                        y = int(

                            HALF_LEDS -

                            (
                                HALF_LEDS -
                                j
                            )

                            *

                            math.cos(
                                math.radians(
                                    degrees_aux
                                )
                            )

                        )


                    # ==========================
                    # CUADRANTE 2
                    # ==========================

                    elif degrees <= 180:

                        degrees_aux = (
                            degrees -
                            90
                        )


                        x = int(

                            HALF_LEDS -

                            (
                                HALF_LEDS -
                                j
                            )

                            *

                            math.sin(
                                math.radians(
                                    degrees_aux
                                )
                            )

                        )


                        y = int(

                            HALF_LEDS -

                            (
                                HALF_LEDS -
                                j
                            )

                            *

                            math.cos(
                                math.radians(
                                    degrees_aux
                                )
                            )

                        )


                    # ==========================
                    # CUADRANTE 3
                    # ==========================

                    elif degrees <= 270:

                        degrees_aux = (
                            270 -
                            degrees
                        )


                        x = int(

                            HALF_LEDS -

                            (
                                HALF_LEDS -
                                j
                            )

                            *

                            math.sin(
                                math.radians(
                                    degrees_aux
                                )
                            )

                        )


                        y = int(

                            HALF_LEDS +

                            (
                                HALF_LEDS -
                                j
                            )

                            *

                            math.cos(
                                math.radians(
                                    degrees_aux
                                )
                            )

                        )


                    # ==========================
                    # CUADRANTE 4
                    # ==========================

                    else:

                        degrees_aux = (
                            degrees -
                            270
                        )


                        x = int(

                            HALF_LEDS +

                            (
                                HALF_LEDS -
                                j
                            )

                            *

                            math.sin(
                                math.radians(
                                    degrees_aux
                                )
                            )

                        )


                        y = int(

                            HALF_LEDS +

                            (
                                HALF_LEDS -
                                j
                            )

                            *

                            math.cos(
                                math.radians(
                                    degrees_aux
                                )
                            )

                        )


                    # ==========================
                    # LIMITES
                    # ==========================

                    if x > NUM_LEDS - 1:

                        x = NUM_LEDS - 1


                    if y > NUM_LEDS - 1:

                        y = NUM_LEDS - 1


                    if x < 0:

                        x = 0


                    if y < 0:

                        y = 0


                    # ==========================
                    # RGB888
                    # ==========================

                    red8 = int(
                        frame[x, y, 2]
                    )


                    green8 = int(
                        frame[x, y, 1]
                    )


                    blue8 = int(
                        frame[x, y, 0]
                    )


                    # ==========================
                    # RGB888 -> RGB565
                    # ==========================

                    red5 = (
                        red8 * 249 + 1014
                    ) >> 11


                    green6 = (
                        green8 * 253 + 505
                    ) >> 10


                    blue5 = (
                        blue8 * 249 + 1014
                    ) >> 11


                    img_converted[row][j] = (

                        (red5 << 11)

                        +

                        (green6 << 5)

                        +

                        blue5

                    )


                degrees += resolution

                row += 1


            # ==================================
            # SEPARADOR ENTRE FRAMES
            # ==================================

            if stored_cont != 0:

                f.write(
                    ",\n"
                )


            # ==================================
            # ESCRIBIR FRAME
            # ==================================

            f.write(
                "{\n"
            )


            for i in range(
                len(img_converted)
            ):


                f.write(
                    "{"
                )


                for j in range(
                    len(img_converted[0])
                ):


                    value = (
                        img_converted[i][j]
                    )


                    hex_value = (
                        "0x{:04X}".format(
                            value
                        )
                    )


                    f.write(
                        hex_value
                    )


                    if (
                        j <
                        len(img_converted[0]) - 1
                    ):

                        f.write(
                            ","
                        )


                f.write(
                    "}"
                )


                if (
                    i <
                    len(img_converted) - 1
                ):

                    f.write(
                        ","
                    )


                f.write(
                    "\n"
                )


            f.write(
                "}"
            )


            stored_cont += 1

            selected_position += 1


        # ======================================
        # SIGUIENTE FRAME DEL VIDEO
        # ======================================

        video_frame_index += 1


        # Ya encontramos todos los necesarios
        if (
            selected_position >=
            len(selected_frames)
        ):

            break


    # ==========================================
    # CERRAR ARRAY
    # ==========================================

    f.write(
        "\n};\n"
    )


    # ==========================================
    # CERRAR ARCHIVOS
    # ==========================================

    f.close()

    cap.release()


    return {
        "total_frames": TOTAL_FRAMES,
        "stored_frames": stored_cont,
        "removed_frames": (
            TOTAL_FRAMES -
            stored_cont
        ),
        "saved_bytes": (
            TOTAL_FRAMES -
            stored_cont
        ) * 12000,
        "fps": FPS_VIDEO,
        "width": WIDTH,
        "height": HEIGHT,
        "selected_frames": selected_frames
    }


def generar_imgsh_animacion(
    origen,
    archivo_dispositivo,
    archivo_drive,
    destino,
    progress=gr.Progress()
):
    try:
        rutas = resolver_entrada(
            origen,
            archivo_dispositivo,
            archivo_drive
        )

        if len(rutas) != 1:
            raise ValueError(
                "Selecciona solamente un video."
            )

        video_path = rutas[0]

        progress(
            0.03,
            desc="Preparando conversión"
        )

        trabajo = crear_trabajo_temporal(
            "imgsh_animacion"
        )

        output_temp = os.path.join(
            trabajo,
            "imgs.h"
        )

        datos = crear_imgsh_animacion_sagrado(
            video_path,
            output_temp,
            progress=progress
        )

        if not os.path.isfile(output_temp):
            raise ValueError(
                "No se generó imgs.h."
            )

        guardar_drive, permitir_descarga = (
            destino_flags(destino)
        )

        ruta_drive = ""

        if guardar_drive:
            progress(
                0.94,
                desc="Guardando imgs.h en Drive"
            )

            ruta_drive = os.path.join(
                SALIDA_DIR,
                "imgs.h"
            )

            shutil.copy2(
                output_temp,
                ruta_drive
            )

        advertencia = ""

        if (
            datos["width"] != 200
            or
            datos["height"] != 200
        ):
            advertencia = (
                "\n\n"
                "⚠️ **Advertencia:** el video no es 200×200. "
                "El conversor original espera videos 200×200. "
                "Para evitar problemas usa primero **Adaptar video**."
            )

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Guardado en Google Drive:**  \n"
                f"`{ruta_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar imgs.h** "
                "para guardarlo en tu dispositivo."
            )

        estado = f"""
### ✅ imgs.h DE ANIMACIÓN GENERADO

**Video:** `{os.path.basename(video_path)}`
**Resolución detectada:** {datos["width"]} × {datos["height"]}
**FPS detectado:** {datos["fps"]:.2f}
**Frames originales:** {datos["total_frames"]}
**Frames almacenados:** {datos["stored_frames"]}
**Frames eliminados:** {datos["removed_frames"]}
**Ahorro aproximado:** {datos["saved_bytes"]:,} bytes

El archivo conserva:

`#define POV_MODE_ANIMATION`
`#define POV_ORIGINAL_FRAMES {datos["total_frames"]}`
`#define POV_STORED_FRAMES {datos["stored_frames"]}`
`const uint16_t frames[][60][100] PROGMEM = {{ ... }};`

{advertencia}
{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            output_temp
            if permitir_descarga
            else None
        )

        return (
            estado,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))


# ============================================================
# imgs.h DESDE IMAGEN
# ============================================================
# Conversión de un BMP normalizado a la matriz polar utilizada para imagen fija.
# Recorrer 360 grados en incrementos de 6 grados para obtener 60 filas.
# Muestrear 100 posiciones radiales por cada ángulo y convertir RGB888 a RGB565.
# Mantener la organización frame5[][100] por compatibilidad directa con firmware.
#
# NUCLEO CRITICO:
# Se conserva la estructura del conversor suministrado para
# imagen fija, especialmente:
#
# const uint16_t frame5[][100] PROGMEM = {
# ============================================================

def crear_imgsh_imagen_sagrado(
    IMAGE_PATH,
    OUTPUT_PATH,
    progress=None
):
    # ==========================================
    # CONFIGURACIÓN
    # ==========================================

    frame = cv2.imread(IMAGE_PATH)

    if frame is None:
        raise Exception("No se pudo leer la imagen")

    alto, ancho = frame.shape[:2]

    if ancho != 200 or alto != 200:
        raise Exception(
            "La imagen no es 200x200. Usa primero ADAPTAR IMAGEN."
        )

    # ==========================================
    # CONVERSIÓN POV
    # ==========================================

    degrees = 0
    degrees_aux = 0

    NUM_LEDS = 200
    HALF_LEDS = int(NUM_LEDS / 2)

    row = 0

    # Paso angular fijo del conversor de imagen: 60 filas por revolución.
    resolution = 6

    img_converted = [
        [0 for k in range(HALF_LEDS)]
        for l in range(int(360 / resolution))
    ]

    total_rows = int(360 / resolution)

    while degrees < 360:

        if progress is not None:
            avance = row / max(1, total_rows)
            progress(
                0.10 + avance * 0.80,
                desc=(
                    "Procesando ángulo "
                    f"{row + 1}/{total_rows}"
                )
            )

        for j in range(0, HALF_LEDS):

            if degrees <= 90:

                degrees_aux = 90 - degrees

                x = int(
                    HALF_LEDS +
                    (HALF_LEDS - j) *
                    math.sin(math.radians(degrees_aux))
                )

                y = int(
                    HALF_LEDS -
                    (HALF_LEDS - j) *
                    math.cos(math.radians(degrees_aux))
                )


            elif degrees <= 180:

                degrees_aux = degrees - 90

                x = int(
                    HALF_LEDS -
                    (HALF_LEDS - j) *
                    math.sin(math.radians(degrees_aux))
                )

                y = int(
                    HALF_LEDS -
                    (HALF_LEDS - j) *
                    math.cos(math.radians(degrees_aux))
                )


            elif degrees <= 270:

                degrees_aux = 270 - degrees

                x = int(
                    HALF_LEDS -
                    (HALF_LEDS - j) *
                    math.sin(math.radians(degrees_aux))
                )

                y = int(
                    HALF_LEDS +
                    (HALF_LEDS - j) *
                    math.cos(math.radians(degrees_aux))
                )


            else:

                degrees_aux = degrees - 270

                x = int(
                    HALF_LEDS +
                    (HALF_LEDS - j) *
                    math.sin(math.radians(degrees_aux))
                )

                y = int(
                    HALF_LEDS +
                    (HALF_LEDS - j) *
                    math.cos(math.radians(degrees_aux))
                )


            if x > NUM_LEDS - 1:
                x = NUM_LEDS - 1

            if y > NUM_LEDS - 1:
                y = NUM_LEDS - 1

            red8 = int(frame[x, y, 2])
            green8 = int(frame[x, y, 1])
            blue8 = int(frame[x, y, 0])

            red5 = (red8 * 249 + 1014) >> 11
            green6 = (green8 * 253 + 505) >> 10
            blue5 = (blue8 * 249 + 1014) >> 11

            img_converted[row][j] = (
                (red5 << 11) +
                (green6 << 5) +
                blue5
            )

        degrees += resolution
        row += 1

    # ==========================================
    # CREAR imgs.h
    # ==========================================

    f = open(OUTPUT_PATH, 'w', encoding='utf-8')

    f.write(
"""//=====================================
// Archivo generado automáticamente
// Conversor POV Imagen
//=====================================

#pragma once
#include <Arduino.h>

const uint16_t frame5[][100] PROGMEM = {

"""
    )

    for i in range(len(img_converted)):

        f.write("{")

        for j in range(len(img_converted[0])):

            value = img_converted[i][j]

            if value == 0:

                f.write("0x0000")


            elif value <= 15:

                h = hex(value)
                f.write(
                    h[:2] + "000" + h[2:]
                )


            elif value <= 255:

                h = hex(value)
                f.write(
                    h[:2] + "00" + h[2:]
                )


            elif value <= 4095:

                h = hex(value)
                f.write(
                    h[:2] + "0" + h[2:]
                )


            else:

                f.write(hex(value))

            f.write(",")

        f.write("},\n")

    f.write("};")

    f.close()

    return {
        "width": ancho,
        "height": alto,
        "rows": len(img_converted),
        "cols": len(img_converted[0])
    }


def generar_imgsh_imagen(
    origen,
    archivo_dispositivo,
    archivo_drive,
    destino,
    progress=gr.Progress()
):
    try:
        rutas = resolver_entrada(
            origen,
            archivo_dispositivo,
            archivo_drive
        )

        if len(rutas) != 1:
            raise ValueError(
                "Selecciona solamente una imagen."
            )

        image_path = rutas[0]

        progress(
            0.03,
            desc="Preparando conversión"
        )

        trabajo = crear_trabajo_temporal(
            "imgsh_imagen"
        )

        output_temp = os.path.join(
            trabajo,
            "imgs.h"
        )

        datos = crear_imgsh_imagen_sagrado(
            image_path,
            output_temp,
            progress=progress
        )

        if not os.path.isfile(output_temp):
            raise ValueError(
                "No se generó imgs.h."
            )

        guardar_drive, permitir_descarga = destino_flags(destino)

        ruta_drive = ""

        if guardar_drive:
            progress(
                0.94,
                desc="Guardando imgs.h en Drive"
            )

            ruta_drive = os.path.join(
                SALIDA_DIR,
                "imgs.h"
            )

            shutil.copy2(
                output_temp,
                ruta_drive
            )

        texto_drive = ""

        if guardar_drive:
            texto_drive = (
                "\n\n"
                "📁 **Guardado en Google Drive:**  \n"
                f"`{ruta_drive}`"
            )

        texto_descarga = ""

        if permitir_descarga:
            texto_descarga = (
                "\n\n"
                "⬇️ Usa **Descargar imgs.h** "
                "para guardarlo en tu dispositivo."
            )

        estado = f"""
### ✅ imgs.h DE IMAGEN GENERADO

**Imagen:** `{os.path.basename(image_path)}`
**Resolución detectada:** {datos['width']} × {datos['height']}
**Matriz generada:** {datos['rows']} filas × {datos['cols']} columnas

El archivo conserva la estructura crítica:

`const uint16_t frame5[][100] PROGMEM = {{ ... }};`

⚠️ Esta función espera una imagen **200×200**. Si tu archivo no tiene ese formato, usa primero **Adaptar imagen**.

{texto_drive}
{texto_descarga}
"""

        progress(
            1.0,
            desc="Listo"
        )

        archivo_descarga = (
            output_temp
            if permitir_descarga
            else None
        )

        return (
            estado,
            archivo_descarga,
            ruta_drive
        )

    except Exception as error:
        raise gr.Error(str(error))


# ============================================================
# CSS
# ============================================================
# Capa exclusivamente visual de POV Creator.
# Definir fondos, tipografías, adaptación a pantalla completa, menús y paneles.
# Mantener esta sección desacoplada de los algoritmos de conversión para permitir
# cambios estéticos sin afectar resultados técnicos ni estructuras imgs.h.

CSS = f"""
@import url('https://fonts.googleapis.com/css2?family=Michroma&family=Orbitron:wght@600;700;800;900&display=swap');

html,
body {{
    margin: 0 !important;
    padding: 0 !important;
    width: 100% !important;
    height: 100% !important;
    overflow: hidden !important;
    background: transparent !important;
}}

.gradio-container {{
    position: fixed !important;
    top: 0 !important;
    left: 0 !important;
    width: 100vw !important;
    height: 100vh !important;
    max-width: none !important;
    margin: 0 !important;
    padding: 0 !important;
    overflow: hidden !important;
    background: transparent !important;
    font-family: 'Orbitron', sans-serif !important;
}}

.gradio-container > .main,
.gradio-container main,
.contain,
.wrap {{
    margin: 0 !important;
    padding: 0 !important;
    max-width: none !important;
    background: transparent !important;
}}

.screen {{
    position: fixed !important;
    top: 0 !important;
    left: 0 !important;
    width: 100vw !important;
    height: 100vh !important;
    margin: 0 !important;
    padding: 0 !important;
    box-sizing: border-box !important;
    overflow: hidden !important;
    background-size: cover !important;
    background-position: center center !important;
    background-repeat: no-repeat !important;
}}

.menu-screen {{
    display: flex !important;
    flex-direction: column !important;
    justify-content: center !important;
    align-items: center !important;
}}

.bg-main {{
    background-image:
        linear-gradient(
            rgba(0,8,30,0.04),
            rgba(0,8,25,0.09)
        ),
        url("{MAIN_DATA}") !important;
}}

.bg-video {{
    background-image:
        linear-gradient(
            rgba(18,0,35,0.03),
            rgba(14,0,30,0.07)
        ),
        url("{VIDEO_DATA}") !important;
}}

.bg-image {{
    background-image:
        linear-gradient(
            rgba(0,20,4,0.03),
            rgba(0,15,3,0.07)
        ),
        url("{IMAGE_DATA}") !important;
}}

.bg-imgsh {{
    background-image:
        linear-gradient(
            rgba(35,0,0,0.03),
            rgba(22,0,0,0.07)
        ),
        url("{IMGSH_DATA}") !important;
}}

.bg-config {{
    background-image:
        linear-gradient(
            rgba(0,0,0,0.03),
            rgba(0,0,0,0.07)
        ),
        url("{CONFIG_DATA}") !important;
}}

.menu-container {{
    width: min(860px, 92vw) !important;
    margin: auto !important;
    padding: 20px !important;
    background: transparent !important;
    border: none !important;
    box-shadow: none !important;
    gap: 14px !important;
}}

/* Fuente anterior del título */
.pov-title {{
    font-family: 'Michroma', sans-serif !important;
    text-align: center;
    text-transform: uppercase;
    color: #FFFFFF;
    line-height: 1.08;
    margin: 0 0 46px 0;
    letter-spacing: 0.10em;
    -webkit-text-stroke: 1.1px rgba(70,155,255,0.95);
    text-shadow:
        0 0 4px white,
        0 0 10px rgba(110,195,255,1),
        0 0 20px rgba(0,105,255,0.82);
}}

.pov-title .top-line {{
    display: block;
    font-size: clamp(42px, 7vw, 78px);
    margin-bottom: 10px;
}}

.pov-title .bottom-line {{
    display: block;
    font-size: clamp(32px, 5vw, 57px);
}}

.section-title {{
    font-family: 'Orbitron', sans-serif !important;
    font-weight: 800;
    text-align: center;
    color: white;
    font-size: clamp(27px, 4vw, 45px);
    letter-spacing: 0.07em;
    margin: 0 0 28px 0;
    -webkit-text-stroke: 0.8px rgba(65,150,255,0.95);
    text-shadow:
        0 0 3px white,
        0 0 9px #55AFFF,
        0 0 16px #006BFF;
}}

.game-menu {{
    width: min(700px, 88vw) !important;
    margin: 5px auto !important;
}}

.game-menu button {{
    width: 100% !important;
    min-height: 58px !important;
    font-family: 'Orbitron', sans-serif !important;
    font-weight: 800 !important;
    color: white !important;
    font-size: clamp(15px, 2vw, 23px) !important;
    text-transform: uppercase;
    background: rgba(15,18,28,0.38) !important;
    border: 1px solid rgba(255,255,255,0.30) !important;
    text-shadow:
        0 0 3px white,
        0 0 8px #5CB7FF,
        0 0 15px #006EFF !important;
    transition: all 0.15s ease !important;
}}

.game-menu button:hover {{
    border-color: white !important;
    background: rgba(255,255,255,0.06) !important;
    transform: scale(1.015);
}}

.back-menu {{
    margin-top: 18px !important;
}}

.work-wrapper {{
    width: 100vw !important;
    height: 100vh !important;
    overflow-y: auto !important;
    padding: 35px 18px 60px 18px !important;
    box-sizing: border-box !important;
}}

.work-panel {{
    width: min(980px, 94vw) !important;
    margin: 0 auto !important;
    padding: 25px !important;
    box-sizing: border-box !important;
    background: rgba(5,7,18,0.62) !important;
    border: 1px solid rgba(255,255,255,0.38) !important;
    border-radius: 10px !important;
    backdrop-filter: blur(6px);
    box-shadow: 0 0 25px rgba(0,0,0,0.28) !important;
}}

.work-panel,
.work-panel label,
.work-panel span,
.work-panel p,
.work-panel div,
.work-panel input,
.work-panel textarea,
.work-panel button {{
    font-family: 'Orbitron', sans-serif !important;
}}

.tool-info {{
    color: white !important;
    opacity: 0.94;
}}

.help-box {{
    margin-top: -6px !important;
    padding: 10px 14px !important;
    background: rgba(0,0,0,0.18) !important;
    border-left: 3px solid rgba(160,220,255,0.75) !important;
    border-radius: 5px !important;
}}

.action-btn button {{
    font-family: 'Orbitron', sans-serif !important;
    font-weight: 900 !important;
    color: white !important;
    background: rgba(0,85,170,0.55) !important;
    border: 1px solid rgba(160,220,255,0.80) !important;
    text-shadow:
        0 0 4px white,
        0 0 10px #168FFF !important;
}}

.action-btn button:hover {{
    background: rgba(0,120,220,0.62) !important;
    border-color: white !important;
}}

.result-panel {{
    margin-top: 18px !important;
    padding: 15px !important;
    background: rgba(0,0,0,0.22) !important;
    border: 1px solid rgba(255,255,255,0.18) !important;
    border-radius: 8px !important;
}}

.manual-panel {{
    line-height: 1.65 !important;
}}

footer {{
    display: none !important;
}}

@media (max-width: 700px) {{
    .work-wrapper {{
        padding: 20px 8px 50px 8px !important;
    }}

    .work-panel {{
        width: 96vw !important;
        padding: 15px !important;
    }}

    .menu-container {{
        width: 94vw !important;
    }}

    .game-menu {{
        width: 90vw !important;
    }}
}}
"""


# ============================================================
# NAVEGACION
# ============================================================
# Control de visibilidad entre módulos dentro de una sola aplicación Gradio.
# Mantener cargados los componentes y alternar únicamente la pantalla activa.
# Evitar recargas completas al cambiar entre animaciones, imágenes, imgs.h y manual.

NUM_PANTALLAS = 10


def mostrar_pantalla(indice):
    return [
        gr.update(
            visible=(i == indice)
        )
        for i in range(NUM_PANTALLAS)
    ]


def ir_principal():
    return mostrar_pantalla(0)


def ir_animaciones():
    return mostrar_pantalla(1)


def ir_imagenes():
    return mostrar_pantalla(2)


def ir_imgs():
    return mostrar_pantalla(3)


def ir_manual():
    return mostrar_pantalla(4)


def ir_stopmotion():
    return mostrar_pantalla(5)


def ir_video():
    return mostrar_pantalla(6)


def ir_adaptar_imagen():
    return mostrar_pantalla(7)


def ir_imgs_animacion():
    return mostrar_pantalla(8)


def ir_imgs_imagen():
    return mostrar_pantalla(9)


# ============================================================
# TEXTO DE AYUDA DE AJUSTE
# ============================================================
# Explicación breve junto al selector para reducir errores de interpretación.
# Presentar la consecuencia visual de cada método sin modificar sus parámetros.

AYUDA_AJUSTE = """
**¿Qué hace cada modo?**

- **RECORTAR:** conserva la proporción y llena todo el cuadro 200×200. Puede cortar parte de los bordes.
- **AJUSTAR:** conserva toda la imagen/video dentro de 200×200. Si sobra espacio, agrega fondo negro.
- **ESTIRAR:** fuerza el contenido directamente a 200×200. No recorta, pero puede deformarlo.
"""


# ============================================================
# INTERFAZ
# ============================================================
# Integración de controles de entrada, destino, previsualización y navegación.
# Permitir selección desde dispositivo o Google Drive y salida como descarga,
# almacenamiento persistente en Drive o ambas opciones simultáneamente.
# Mantener la lógica de procesamiento en funciones separadas de la presentación.

with gr.Blocks(
    css=CSS,
    title="POV Creator"
) as app:

    # ========================================================
    # 0 - PRINCIPAL
    # ========================================================

    with gr.Column(
        visible=True,
        elem_classes=[
            "screen",
            "menu-screen",
            "bg-main"
        ]
    ) as panel_principal:

        with gr.Column(
            elem_classes=[
                "menu-container"
            ]
        ):

            gr.HTML(
                """
                <div class="pov-title">
                    <span class="top-line">POV</span>
                    <span class="bottom-line">CREATOR</span>
                </div>
                """
            )

            btn_animaciones = gr.Button(
                "ANIMACIONES",
                elem_classes=["game-menu"]
            )

            btn_imagenes = gr.Button(
                "IMÁGENES",
                elem_classes=["game-menu"]
            )

            btn_imgs = gr.Button(
                "GENERAR imgs.h",
                elem_classes=["game-menu"]
            )

            btn_manual = gr.Button(
                "MANUAL",
                elem_classes=["game-menu"]
            )


    # ========================================================
    # 1 - MENU ANIMACIONES
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "menu-screen",
            "bg-video"
        ]
    ) as panel_animaciones:

        with gr.Column(
            elem_classes=[
                "menu-container"
            ]
        ):

            gr.HTML(
                """
                <div class="section-title">
                    ANIMACIONES
                </div>
                """
            )

            btn_stopmotion = gr.Button(
                "CREAR ANIMACIÓN",
                elem_classes=["game-menu"]
            )

            btn_video = gr.Button(
                "ADAPTAR VIDEO EXISTENTE",
                elem_classes=["game-menu"]
            )

            btn_back_animaciones = gr.Button(
                "VOLVER",
                elem_classes=[
                    "game-menu",
                    "back-menu"
                ]
            )


    # ========================================================
    # 2 - MENU IMAGENES
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "menu-screen",
            "bg-image"
        ]
    ) as panel_imagenes:

        with gr.Column(
            elem_classes=[
                "menu-container"
            ]
        ):

            gr.HTML(
                """
                <div class="section-title">
                    IMÁGENES
                </div>
                """
            )

            btn_adaptar_imagen = gr.Button(
                "ADAPTAR IMAGEN",
                elem_classes=["game-menu"]
            )

            btn_back_imagenes = gr.Button(
                "VOLVER",
                elem_classes=[
                    "game-menu",
                    "back-menu"
                ]
            )


    # ========================================================
    # 3 - MENU imgs.h
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "menu-screen",
            "bg-imgsh"
        ]
    ) as panel_imgs:

        with gr.Column(
            elem_classes=[
                "menu-container"
            ]
        ):

            gr.HTML(
                """
                <div class="section-title">
                    GENERAR imgs.h
                </div>
                """
            )

            btn_imgs_animacion = gr.Button(
                "DESDE ANIMACIÓN",
                elem_classes=["game-menu"]
            )

            btn_imgs_imagen = gr.Button(
                "DESDE IMAGEN",
                elem_classes=["game-menu"]
            )

            btn_back_imgs = gr.Button(
                "VOLVER",
                elem_classes=[
                    "game-menu",
                    "back-menu"
                ]
            )


    # ========================================================
    # 4 - MANUAL
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-config"
        ]
    ) as panel_manual:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel",
                    "manual-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        MANUAL
                    </div>
                    """
                )

                # ------------------------------------------------
                # INFORMACION QUE YA ESTABA EN CONFIGURACION
                # ------------------------------------------------

                gr.Markdown(
                    """
### Parámetros actuales del sistema

**Resolución POV:** 200 × 200 🔒
**Máximo de frames almacenados:** 69 🔒
**Formato interno:** RGB565 🔒
**FPS recomendado de preparación:** 10

---

### ESP32 y OTA

El proyecto utiliza una **ESP32-WROOM-32 de 4 MB** y la tabla de particiones:

**`No FS 4MB (2MB APP x2)`**

Esta configuración reserva aproximadamente **2 MB para cada aplicación** y permite mantener **OTA**. El tamaño máximo de aplicación que hemos venido usando es aproximadamente **2.031.616 bytes**.

Para conservar OTA:

- No usar una tabla **No OTA**.
- No cambiar a **Huge APP** para este flujo.
- Si cambias la tabla de particiones en Arduino IDE, debes hacer **una carga por USB** para escribir la nueva tabla; después puedes volver a cargar por OTA.
- Cada frame RGB565 del POV ocupa aproximadamente **12.000 bytes**: 60 filas × 100 valores × 2 bytes.
- El límite actual de **69 frames** protege el espacio disponible para el firmware y las librerías Wi‑Fi/OTA.
- La versión de animación que hemos probado con 69 frames ocupa aproximadamente **1.919.839 bytes**, alrededor del 94 % de la partición de aplicación. Por eso no conviene aumentar el número de frames sin volver a comprobar el tamaño compilado.

---

### Guía rápida de POV Creator

#### 1. Animaciones → Crear Animación

Usa entre **1 y 69 imágenes**. El sistema las adapta a 200×200 y las distribuye hasta crear exactamente **69 frames**, 10 FPS, H.264 y YUV420P.

Puedes elegir:

**Origen**
- Dispositivo: arrastrar archivos o buscarlos.
- Google Drive: seleccionar archivos dentro de `TESIS_POV`.

**Destino**
- Descargar al dispositivo.
- Guardar en Google Drive.
- Drive + descarga.

En Drive, el MP4 queda en:

`TESIS_POV/Videos/`

y las imágenes intermedias 200×200 quedan en:

`TESIS_POV/StopMotion/<Nombre>/Imagenes_200x200/`

#### 2. Animaciones → Adaptar video existente

Convierte un video normal a:

**MP4 · 200×200 · 10 FPS · H.264 · YUV420P · sin audio**

El video conserva aproximadamente su duración. Si tiene más de 69 frames, esa reducción se hace después al generar `imgs.h`.

#### 3. Imágenes → Adaptar imagen

Convierte JPG, PNG, WEBP, BMP y otros formatos admitidos por Pillow a:

**BMP · 200×200 · RGB**

En Drive queda en:

`TESIS_POV/Imagenes/`

#### 4. Generar imgs.h → Desde animación

Selecciona preferiblemente uno de los MP4 que ya salió de **Crear Animación** o **Adaptar video**.

El conversor conserva como máximo **69 frames** y genera exactamente la estructura que utiliza el firmware de animación:

`const uint16_t frames[][60][100] PROGMEM = { ... };`

Si eliges Drive como destino se guarda en:

`TESIS_POV/Salida/imgs.h`

#### 5. Arduino IDE

1. Sustituye el `imgs.h` del proyecto por el generado.
2. Comprueba la placa ESP32 y la tabla `No FS 4MB (2MB APP x2)`.
3. Compila y verifica que el sketch no supere el espacio disponible.
4. Con la ESP32 y el PC en la misma Wi‑Fi, selecciona `POV-ESP32` y realiza la carga OTA.
5. Si acabas de cambiar la tabla de particiones, realiza primero una carga por USB.

> **Importante:** el generador `imgs.h` de animación es una parte crítica del proyecto. Su geometría, formato RGB565, selección de frames y estructura `frames[][60][100]` no deben modificarse sin validar primero el firmware.
                    """
                )

                btn_back_manual = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 5 - STOP MOTION
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-video"
        ]
    ) as panel_stopmotion:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        CREAR ANIMACIÓN
                    </div>
                    """
                )

                gr.Markdown(
                    """
**Resultado:** MP4 · 200×200 · 10 FPS · 69 frames · H.264 · YUV420P
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    stop_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Dispositivo",
                        label="1. Origen de las imágenes"
                    )

                    stop_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino del resultado"
                    )

                stop_files_device = gr.File(
                    label=(
                        "Arrastra aquí entre 1 y 69 imágenes "
                        "o haz clic para buscarlas"
                    ),
                    file_count="multiple",
                    file_types=[
                        "image",
                        ".bmp",
                        ".webp",
                        ".tif",
                        ".tiff"
                    ],
                    type="filepath",
                    visible=True
                )

                stop_files_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="multiple",
                    label=(
                        "Seleccionar imágenes desde TESIS_POV"
                    ),
                    height=280,
                    visible=False
                )

                stop_mode = gr.Dropdown(
                    choices=[
                        "RECORTAR",
                        "AJUSTAR",
                        "ESTIRAR"
                    ],
                    value="RECORTAR",
                    label="Modo de ajuste 200×200"
                )

                gr.Markdown(
                    AYUDA_AJUSTE,
                    elem_classes=["help-box"]
                )

                stop_name = gr.Textbox(
                    label="Nombre del video",
                    placeholder="Ejemplo: Bienvenida"
                )

                stop_generate = gr.Button(
                    "GENERAR ANIMACIÓN",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    stop_status = gr.Markdown()

                    stop_preview = gr.Video(
                        label="Vista previa del MP4"
                    )

                    stop_download = gr.File(
                        label="Descargar MP4",
                        interactive=False
                    )

                    stop_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_stop = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 6 - ADAPTAR VIDEO
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-video"
        ]
    ) as panel_video:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        ADAPTAR VIDEO
                    </div>
                    """
                )

                gr.Markdown(
                    """
**Resultado:** MP4 · 200×200 · 10 FPS · H.264 · YUV420P · sin audio
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    video_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Dispositivo",
                        label="1. Origen del video"
                    )

                    video_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino del resultado"
                    )

                video_file_device = gr.File(
                    label=(
                        "Arrastra el video aquí "
                        "o haz clic para buscarlo"
                    ),
                    file_count="single",
                    file_types=[
                        "video",
                        ".mp4",
                        ".mov",
                        ".avi",
                        ".mkv",
                        ".webm",
                        ".m4v",
                        ".mpeg",
                        ".mpg",
                        ".wmv",
                        ".flv",
                        ".3gp",
                        ".ts",
                        ".mts",
                        ".m2ts"
                    ],
                    type="filepath",
                    visible=True
                )

                video_file_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="single",
                    label=(
                        "Seleccionar video desde TESIS_POV"
                    ),
                    height=280,
                    visible=False
                )

                video_mode = gr.Dropdown(
                    choices=[
                        "RECORTAR",
                        "AJUSTAR",
                        "ESTIRAR"
                    ],
                    value="RECORTAR",
                    label="Modo de ajuste 200×200"
                )

                gr.Markdown(
                    AYUDA_AJUSTE,
                    elem_classes=["help-box"]
                )

                video_name = gr.Textbox(
                    label="Nombre del video final",
                    placeholder="Ejemplo: MinecraftSteve"
                )

                video_generate = gr.Button(
                    "CONVERTIR VIDEO",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    video_status = gr.Markdown()

                    video_preview = gr.Video(
                        label="Vista previa del MP4"
                    )

                    video_download = gr.File(
                        label="Descargar MP4",
                        interactive=False
                    )

                    video_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_video = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 7 - ADAPTAR IMAGEN
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-image"
        ]
    ) as panel_adaptar_imagen:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        ADAPTAR IMAGEN
                    </div>
                    """
                )

                gr.Markdown(
                    """
**Resultado:** BMP · 200×200 · RGB
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    image_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Dispositivo",
                        label="1. Origen de la imagen"
                    )

                    image_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino del resultado"
                    )

                image_file_device = gr.File(
                    label=(
                        "Arrastra una imagen aquí "
                        "o haz clic para buscarla"
                    ),
                    file_count="single",
                    file_types=[
                        "image",
                        ".bmp",
                        ".webp",
                        ".tif",
                        ".tiff"
                    ],
                    type="filepath",
                    visible=True
                )

                image_file_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="single",
                    label=(
                        "Seleccionar imagen desde TESIS_POV"
                    ),
                    height=280,
                    visible=False
                )

                image_mode = gr.Dropdown(
                    choices=[
                        "RECORTAR",
                        "AJUSTAR",
                        "ESTIRAR"
                    ],
                    value="RECORTAR",
                    label="Modo de ajuste 200×200"
                )

                gr.Markdown(
                    AYUDA_AJUSTE,
                    elem_classes=["help-box"]
                )

                image_name = gr.Textbox(
                    label="Nombre del BMP final",
                    placeholder="Ejemplo: Logo_USC"
                )

                image_generate = gr.Button(
                    "CONVERTIR IMAGEN",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    image_status = gr.Markdown()

                    with gr.Row():

                        image_original_preview = gr.Image(
                            label="Original"
                        )

                        image_result_preview = gr.Image(
                            label="Resultado 200×200 BMP"
                        )

                    image_download = gr.File(
                        label="Descargar BMP",
                        interactive=False
                    )

                    image_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_adaptar_imagen = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 8 - imgs.h DESDE ANIMACION
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-imgsh"
        ]
    ) as panel_imgs_animacion:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        imgs.h — ANIMACIÓN
                    </div>
                    """
                )

                gr.Markdown(
                    """
Selecciona preferiblemente un MP4 creado previamente por **Crear Animación** o **Adaptar video**.

El núcleo del conversor mantiene sin cambios la estructura crítica:

`const uint16_t frames[][60][100] PROGMEM = { ... };`
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    imgsh_anim_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Google Drive",
                        label="1. Origen de la animación"
                    )

                    imgsh_anim_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino de imgs.h"
                    )

                imgsh_anim_file_device = gr.File(
                    label=(
                        "Arrastra el MP4 aquí "
                        "o haz clic para buscarlo"
                    ),
                    file_count="single",
                    file_types=[
                        "video",
                        ".mp4"
                    ],
                    type="filepath",
                    visible=False
                )

                imgsh_anim_file_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="single",
                    label=(
                        "Seleccionar MP4 desde TESIS_POV"
                    ),
                    height=280,
                    visible=True
                )

                imgsh_anim_generate = gr.Button(
                    "GENERAR imgs.h",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    imgsh_anim_status = gr.Markdown()

                    imgsh_anim_download = gr.File(
                        label="Descargar imgs.h",
                        interactive=False
                    )

                    imgsh_anim_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_imgs_animacion = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # 9 - imgs.h DESDE IMAGEN
    # ========================================================

    with gr.Column(
        visible=False,
        elem_classes=[
            "screen",
            "bg-imgsh"
        ]
    ) as panel_imgs_imagen:

        with gr.Column(
            elem_classes=[
                "work-wrapper"
            ]
        ):

            with gr.Column(
                elem_classes=[
                    "work-panel"
                ]
            ):

                gr.HTML(
                    """
                    <div class="section-title">
                        imgs.h — IMAGEN
                    </div>
                    """
                )

                gr.Markdown(
                    """
Usa preferiblemente un **BMP 200×200** generado previamente por **Adaptar imagen**.

El núcleo del conversor mantiene sin cambios la estructura crítica:

`const uint16_t frame5[][100] PROGMEM = { ... };`
                    """,
                    elem_classes=[
                        "tool-info"
                    ]
                )

                with gr.Row():

                    imgsh_image_origen = gr.Radio(
                        choices=[
                            "Dispositivo",
                            "Google Drive"
                        ],
                        value="Google Drive",
                        label="1. Origen de la imagen"
                    )

                    imgsh_image_destino = gr.Radio(
                        choices=[
                            "Descargar al dispositivo",
                            "Google Drive",
                            "Drive + descarga"
                        ],
                        value="Google Drive",
                        label="2. Destino de imgs.h"
                    )

                imgsh_image_file_device = gr.File(
                    label=(
                        "Arrastra la imagen aquí "
                        "o haz clic para buscarla"
                    ),
                    file_count="single",
                    file_types=[
                        "image",
                        ".bmp",
                        ".png",
                        ".jpg",
                        ".jpeg",
                        ".webp",
                        ".tif",
                        ".tiff"
                    ],
                    type="filepath",
                    visible=False
                )

                imgsh_image_file_drive = gr.FileExplorer(
                    root_dir=BASE_DIR,
                    glob="**/*",
                    file_count="single",
                    label=(
                        "Seleccionar imagen desde TESIS_POV"
                    ),
                    height=280,
                    visible=True
                )

                imgsh_image_generate = gr.Button(
                    "GENERAR imgs.h",
                    elem_classes=["action-btn"]
                )

                with gr.Column(
                    elem_classes=[
                        "result-panel"
                    ]
                ):

                    imgsh_image_status = gr.Markdown()

                    imgsh_image_download = gr.File(
                        label="Descargar imgs.h",
                        interactive=False
                    )

                    imgsh_image_drive_path = gr.Textbox(
                        label="Ruta final en Google Drive",
                        interactive=False
                    )

                btn_back_imgs_imagen = gr.Button(
                    "VOLVER",
                    elem_classes=[
                        "game-menu",
                        "back-menu"
                    ]
                )


    # ========================================================
    # LISTA DE PANELES
    # ========================================================

    PANELES = [
        panel_principal,
        panel_animaciones,
        panel_imagenes,
        panel_imgs,
        panel_manual,
        panel_stopmotion,
        panel_video,
        panel_adaptar_imagen,
        panel_imgs_animacion,
        panel_imgs_imagen
    ]


    # ========================================================
    # NAVEGACION PRINCIPAL
    # ========================================================

    btn_animaciones.click(
        ir_animaciones,
        outputs=PANELES
    )

    btn_imagenes.click(
        ir_imagenes,
        outputs=PANELES
    )

    btn_imgs.click(
        ir_imgs,
        outputs=PANELES
    )

    btn_manual.click(
        ir_manual,
        outputs=PANELES
    )


    # ========================================================
    # ANIMACIONES
    # ========================================================

    btn_stopmotion.click(
        ir_stopmotion,
        outputs=PANELES
    )

    btn_video.click(
        ir_video,
        outputs=PANELES
    )

    btn_back_animaciones.click(
        ir_principal,
        outputs=PANELES
    )

    stop_origen.change(
        mostrar_origen,
        inputs=[stop_origen],
        outputs=[
            stop_files_device,
            stop_files_drive
        ]
    )

    stop_generate.click(
        generar_stop_motion,
        inputs=[
            stop_origen,
            stop_files_device,
            stop_files_drive,
            stop_destino,
            stop_mode,
            stop_name
        ],
        outputs=[
            stop_status,
            stop_preview,
            stop_download,
            stop_drive_path
        ]
    )

    video_origen.change(
        mostrar_origen,
        inputs=[video_origen],
        outputs=[
            video_file_device,
            video_file_drive
        ]
    )

    video_generate.click(
        convertir_video_pov,
        inputs=[
            video_origen,
            video_file_device,
            video_file_drive,
            video_destino,
            video_mode,
            video_name
        ],
        outputs=[
            video_status,
            video_preview,
            video_download,
            video_drive_path
        ]
    )

    btn_back_stop.click(
        ir_animaciones,
        outputs=PANELES
    )

    btn_back_video.click(
        ir_animaciones,
        outputs=PANELES
    )


    # ========================================================
    # IMAGENES
    # ========================================================

    btn_adaptar_imagen.click(
        ir_adaptar_imagen,
        outputs=PANELES
    )

    btn_back_imagenes.click(
        ir_principal,
        outputs=PANELES
    )

    image_origen.change(
        mostrar_origen,
        inputs=[image_origen],
        outputs=[
            image_file_device,
            image_file_drive
        ]
    )

    image_generate.click(
        convertir_imagen_interfaz,
        inputs=[
            image_origen,
            image_file_device,
            image_file_drive,
            image_destino,
            image_mode,
            image_name
        ],
        outputs=[
            image_status,
            image_original_preview,
            image_result_preview,
            image_download,
            image_drive_path
        ]
    )

    btn_back_adaptar_imagen.click(
        ir_imagenes,
        outputs=PANELES
    )


    # ========================================================
    # imgs.h
    # ========================================================

    btn_imgs_animacion.click(
        ir_imgs_animacion,
        outputs=PANELES
    )

    btn_imgs_imagen.click(
        ir_imgs_imagen,
        outputs=PANELES
    )

    btn_back_imgs.click(
        ir_principal,
        outputs=PANELES
    )

    imgsh_anim_origen.change(
        mostrar_origen,
        inputs=[imgsh_anim_origen],
        outputs=[
            imgsh_anim_file_device,
            imgsh_anim_file_drive
        ]
    )

    imgsh_anim_generate.click(
        generar_imgsh_animacion,
        inputs=[
            imgsh_anim_origen,
            imgsh_anim_file_device,
            imgsh_anim_file_drive,
            imgsh_anim_destino
        ],
        outputs=[
            imgsh_anim_status,
            imgsh_anim_download,
            imgsh_anim_drive_path
        ]
    )

    btn_back_imgs_animacion.click(
        ir_imgs,
        outputs=PANELES
    )

    imgsh_image_origen.change(
        mostrar_origen,
        inputs=[imgsh_image_origen],
        outputs=[
            imgsh_image_file_device,
            imgsh_image_file_drive
        ]
    )

    imgsh_image_generate.click(
        generar_imgsh_imagen,
        inputs=[
            imgsh_image_origen,
            imgsh_image_file_device,
            imgsh_image_file_drive,
            imgsh_image_destino
        ],
        outputs=[
            imgsh_image_status,
            imgsh_image_download,
            imgsh_image_drive_path
        ]
    )

    btn_back_imgs_imagen.click(
        ir_imgs,
        outputs=PANELES
    )


    # ========================================================
    # MANUAL
    # ========================================================

    btn_back_manual.click(
        ir_principal,
        outputs=PANELES
    )


# ============================================================
# INICIAR APP
# ============================================================
# Publicar la interfaz Gradio desde el entorno de Colab.
# inline=False evita incrustar la aplicación debajo de la celda y favorece uso
# en una pestaña independiente. El enlace compartido permanece ligado a la sesión.

resultado = app.launch(
    share=True,
    inline=False,
    prevent_thread_lock=True,
    show_error=True,
    max_file_size="500mb"
)


# ============================================================
# ENCONTRAR URL PUBLICA
# ============================================================
# Recuperar la dirección temporal generada por Gradio para abrir POV Creator.
# La dirección deja de funcionar al finalizar o reiniciar la sesión de Colab.

share_url = getattr(
    app,
    "share_url",
    None
)

if not share_url:
    try:
        if isinstance(resultado, tuple):
            for elemento in resultado:
                if (
                    isinstance(elemento, str)
                    and
                    "gradio.live" in elemento
                ):
                    share_url = elemento
                    break
    except Exception:
        pass


# ============================================================
# BOTON PARA ABRIR
# ============================================================
# Presentar un acceso directo desde el notebook hacia la interfaz independiente.
# Mantener el notebook como entorno de ejecución y la pestaña Gradio como interfaz.

if share_url:
    print()
    print("==============================================")
    print(" POV CREATOR ACTIVO")
    print("==============================================")
    print()
    print(share_url)

    display(
        HTML(
            f"""
            <div
                style="
                    text-align:center;
                    margin:25px;
                "
            >
                <a
                    href="{share_url}"
                    target="_blank"
                    style="
                        display:inline-block;
                        padding:16px 34px;
                        background:
                            linear-gradient(
                                90deg,
                                #073E82,
                                #0099E8
                            );
                        color:white;
                        text-decoration:none;
                        font-family:Arial,sans-serif;
                        font-weight:bold;
                        font-size:16px;
                        border-radius:8px;
                        box-shadow:
                            0 0 18px
                            rgba(0,150,255,0.65);
                    "
                >
                    🚀 ABRIR POV CREATOR
                </a>
            </div>
            """
        )
    )

else:
    print()
    print("✅ POV Creator está ejecutándose.")
    print(
        "Abre manualmente el enlace "
        "gradio.live mostrado arriba."
    )

Mounted at /content/drive

 VALIDANDO ASSETS

✅ Main
✅ Video
✅ Image
✅ imgsh
✅ Config


/tmp/ipykernel_1058/3838987354.py:3085: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a308adbef7efee54b6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)

 POV CREATOR ACTIVO

https://a308adbef7efee54b6.gradio.live
